# 04_03 - Dataset de modelado SER y baseline temporal

Este notebook construye el dataset de modelado SER a partir del panel `SER_barrio_intervalo` generado en `04_02_ser_panel_barrio_intervalo.ipynb` y evalúa una comparación mínima entre un baseline histórico y un modelo mínimo que combina información histórica con variables disponibles antes del intervalo objetivo.

El objetivo no es predecir la ocupación real de todas las plazas SER, porque el TFM no observa dicha ocupación. El target utilizado es `ocupacion_pagada_proxy`, una medida histórica de presión pagada SER construida a partir de minutos pagados solapados y capacidad-tiempo disponible por barrio e intervalo. Por tanto, los resultados deben interpretarse como estimación de un proxy histórico de dificultad, no como disponibilidad real plaza a plaza.

El notebook mantiene la separación metodológica entre target, métricas diagnósticas y variables predictoras. Las métricas SER contemporáneas del intervalo objetivo no se usan como features, ya que derivan de los mismos tiques que construyen el target y, además, no estarían disponibles en una predicción futura real.

## 0. Configuración, objetivo y contrato metodológico

El objetivo de este notebook es construir una tabla de modelado reproducible y evaluar si un modelo mínimo con variables disponibles ex ante mejora a un baseline histórico basado en patrones previos del propio panel SER.

El contrato metodológico queda fijado en los siguientes puntos:

* La unidad analítica es `barrio_key × intervalo_inicio`.
* La granularidad temporal final es de 30 minutos.
* El target es `ocupacion_pagada_proxy`.
* El target representa presión pagada SER observada históricamente, no ocupación real total ni probabilidad real de encontrar plaza.
* El baseline histórico aprende perfiles del pasado y sirve como referencia mínima.
* El modelo mínimo incorpora el componente histórico del baseline y añade variables ex ante.
* Ninguna feature puede usar información posterior al intervalo objetivo.
* Ninguna feature agregada puede calcularse usando filas de validación o test.
* Las métricas SER contemporáneas del intervalo objetivo se conservan solo para diagnóstico, no como predictores.
* La validación es temporal; no se utiliza validación cruzada aleatoria.

El notebook admite dos modos de ejecución. El modo `smoke` permite validar técnicamente el pipeline con una muestra reducida, sin interpretar resultados ni escribir salidas de producción. El modo `full` ejecuta el flujo completo sobre el panel final y genera resultados interpretables, aunque la escritura de artefactos queda protegida por `WRITE_OUTPUTS`. En la ejecución documentada en este notebook se utiliza `RUN_MODE = "full"`, `WRITE_OUTPUTS = True` y `OVERWRITE_OUTPUTS = True`, por lo que los resultados son interpretables y las salidas finales se regeneran en disco.

In [12]:
from pathlib import Path
import json
import sys

from IPython.display import display
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError("No se encontró data_catalog.csv en la jerarquía de directorios.")


ROOT = find_repo_root()

RUN_MODE = "full"
WRITE_OUTPUTS = True
OVERWRITE_OUTPUTS = True
SHOW_DEBUG_TABLES = False

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError(f"RUN_MODE no válido: {RUN_MODE!r}")

PANEL_FINAL_PATH = ROOT / "data/processed/core/ser/ser_barrio_intervalo_global_final.parquet"
BARRIO_CAPACITY_PATH = ROOT / "data/processed/core/ser/ser_barrio_capacidad_anio.parquet"
CALENDAR_PATH = ROOT / "data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet"
MODELING_DIR = ROOT / "data/processed/core/ser/modeling"
REPORTS_TABLES_DIR = ROOT / "reports" / "tables"
REPORTS_FIGURES_DIR = ROOT / "reports" / "figures"

TARGET_COLUMN = "ocupacion_pagada_proxy"
KEY_COLUMNS = ["barrio_key", "intervalo_inicio"]
LEAKAGE_COLUMNS = [
    "n_tiques_solapados",
    "minutos_solapados",
    "stock_inicio_proxy",
    "stock_fin_proxy",
    "afluencia_tiques",
    "liberacion_tiques",
    "persistencia_completa",
    "saldo_flujo",
    "rotacion_bruta",
]
FORBIDDEN_FEATURE_COLUMNS = [TARGET_COLUMN] + LEAKAGE_COLUMNS

initial_config = pd.DataFrame([
    {"item": "ROOT", "value": str(ROOT), "exists": ROOT.exists()},
    {"item": "RUN_MODE", "value": RUN_MODE, "exists": pd.NA},
    {"item": "WRITE_OUTPUTS", "value": WRITE_OUTPUTS, "exists": pd.NA},
    {"item": "OVERWRITE_OUTPUTS", "value": OVERWRITE_OUTPUTS, "exists": pd.NA},
    {"item": "PANEL_FINAL_PATH", "value": str(PANEL_FINAL_PATH.relative_to(ROOT)), "exists": PANEL_FINAL_PATH.exists()},
    {"item": "BARRIO_CAPACITY_PATH", "value": str(BARRIO_CAPACITY_PATH.relative_to(ROOT)), "exists": BARRIO_CAPACITY_PATH.exists()},
    {"item": "CALENDAR_PATH", "value": str(CALENDAR_PATH.relative_to(ROOT)), "exists": CALENDAR_PATH.exists()},
])
display(initial_config)

,item,value,exists
0,ROOT,/Users/hugo/TFM_parking_madrid,True
1,RUN_MODE,full,<NA>
2,WRITE_OUTPUTS,True,<NA>
3,OVERWRITE_OUTPUTS,True,<NA>
4,PANEL_FINAL_PATH,data/processed/core/ser/ser_barrio_intervalo_g...,True
5,BARRIO_CAPACITY_PATH,data/processed/core/ser/ser_barrio_capacidad_a...,True
6,CALENDAR_PATH,data/interim/contexto/contexto_calendario_labo...,True


## 1. Entradas, salidas previstas y modos de ejecución

La entrada principal es:

* `data/processed/core/ser/ser_barrio_intervalo_global_final.parquet`

Esta tabla contiene el panel final SER por barrio e intervalo a 30 minutos, incluyendo el target `ocupacion_pagada_proxy` y métricas diagnósticas de uso pagado.

Como entradas auxiliares se consideran:

* `data/processed/core/ser/ser_barrio_capacidad_anio.parquet`
* `data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet`

La capacidad anual por barrio se utiliza como control estructural y como variable ex ante cuando está presente en el panel final. El calendario laboral se mantiene como entrada auxiliar de referencia y control de disponibilidad, aunque en esta versión del notebook no se une directamente al dataset de modelado: las variables temporales utilizadas se derivan de `intervalo_inicio`, teniendo en cuenta que el panel ya se limita a ventanas de régimen SER observable.

Las fuentes de presión estructural SER, como autorizaciones o IVTM, no se incorporan en este notebook. Su integración se reserva para un notebook posterior de escenarios, donde podrán utilizarse como componente estructural de un índice ajustado de dificultad, no como predictores contemporáneos del target pagado.

El notebook distingue dos modos de ejecución:

* `RUN_MODE = "smoke"`: ejecución técnica reducida, no representativa, sin escritura de outputs de producción.
* `RUN_MODE = "full"`: ejecución completa con splits temporales definitivos y escritura controlada de salidas reutilizables.

Solo se guardarán salidas con uso posterior claro: métricas comparativas para memoria, predicciones evaluadas para mapas o agregaciones posteriores, y artefactos compactos del modelo M0 seleccionado para escenarios futuros. No se guardará una copia completa del dataset de modelado, ya que puede reconstruirse desde el panel final y no constituye por sí mismo un modelo reutilizable.

In [13]:
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq


def relpath(path: Path | str) -> str:
    path = Path(path)
    try:
        return str(path.resolve().relative_to(ROOT))
    except ValueError:
        return str(path)


def safe_parquet_metadata(path: Path | str) -> dict:
    path = Path(path)
    result = {
        "path": relpath(path),
        "exists": path.exists(),
        "is_dir": path.is_dir() if path.exists() else False,
        "n_files": pd.NA,
        "n_rows_metadata": pd.NA,
        "n_columns_metadata": pd.NA,
        "size_mb": pd.NA,
        "columns_preview": [],
        "read_error": pd.NA,
    }

    if not result["exists"]:
        result["read_error"] = "path_not_found"
        return result

    try:
        if path.is_dir():
            parquet_files = sorted(path.rglob("*.parquet"))
            result["n_files"] = len(parquet_files)
            result["size_mb"] = round(sum(file.stat().st_size for file in parquet_files) / 1024**2, 3)
            if parquet_files:
                dataset = ds.dataset(path, format="parquet")
                columns = dataset.schema.names
                result["n_columns_metadata"] = len(columns)
                result["columns_preview"] = columns[:20]
        else:
            result["n_files"] = 1
            result["size_mb"] = round(path.stat().st_size / 1024**2, 3)
            if path.suffix.lower() == ".parquet":
                parquet_file = pq.ParquetFile(path)
                columns = parquet_file.schema_arrow.names
                result["n_rows_metadata"] = parquet_file.metadata.num_rows
                result["n_columns_metadata"] = len(columns)
                result["columns_preview"] = columns[:20]
            else:
                result["read_error"] = "not_a_parquet_path"
    except Exception as exc:
        result["read_error"] = f"{type(exc).__name__}: {exc}"

    return result


input_sources = pd.DataFrame([
    {
        "dataset_id": "ser_barrio_intervalo_global_final",
        "path": relpath(PANEL_FINAL_PATH),
        "exists": PANEL_FINAL_PATH.exists(),
        "critical": True,
        "role": "entrada crítica, fuente nuclear de target y métricas diagnósticas",
        "use_in_04_03": "dataset base de modelado SER",
    },
    {
        "dataset_id": "ser_barrio_capacidad_anio",
        "path": relpath(BARRIO_CAPACITY_PATH),
        "exists": BARRIO_CAPACITY_PATH.exists(),
        "critical": True,
        "role": "entrada auxiliar crítica, control estructural y posible feature ex ante",
        "use_in_04_03": "validación estructural y feature ex ante potencial",
    },
    {
        "dataset_id": "contexto_calendario_laboral_clean",
        "path": relpath(CALENDAR_PATH),
        "exists": CALENDAR_PATH.exists(),
        "critical": True,
        "role": "entrada auxiliar crítica, variables temporales lógicas",
        "use_in_04_03": "derivación de variables temporales lógicas",
    },
])
if SHOW_DEBUG_TABLES:
    display(input_sources)

missing_critical_inputs = input_sources.loc[input_sources["critical"] & ~input_sources["exists"]]
if not missing_critical_inputs.empty:
    display(missing_critical_inputs)
    missing_ids = missing_critical_inputs["dataset_id"].tolist()
    raise FileNotFoundError(f"Faltan entradas críticas para 04_03: {missing_ids}")

input_metadata = pd.DataFrame([
    safe_parquet_metadata(PANEL_FINAL_PATH),
    safe_parquet_metadata(BARRIO_CAPACITY_PATH),
    safe_parquet_metadata(CALENDAR_PATH),
])
if SHOW_DEBUG_TABLES:
    display(input_metadata)

run_modes_table = pd.DataFrame([
    {
        "run_mode": "smoke",
        "purpose": "prueba técnica reducida",
        "representative_results": False,
        "writes_outputs": False,
        "expected_use": "valida estructura",
    },
    {
        "run_mode": "full",
        "purpose": "ejecución completa",
        "representative_results": True,
        "writes_outputs": "escritura controlada si WRITE_OUTPUTS=True",
        "expected_use": "resultados interpretables",
    },
])
display(run_modes_table)

output_plan = pd.DataFrame([
    {
        "output_id": "ser_model_predictions.parquet",
        "path": relpath(MODELING_DIR / "ser_model_predictions.parquet"),
        "write_in_smoke": False,
        "write_in_full": "RUN_MODE='full' y WRITE_OUTPUTS=True",
        "created_by_04_03": True,
        "use_after_04_03": "mapa posterior y agregaciones por barrio/franja",
    },
    {
        "output_id": "ser_model_metrics_comparison.csv",
        "path": relpath(REPORTS_TABLES_DIR / "ser_model_metrics_comparison.csv"),
        "write_in_smoke": False,
        "write_in_full": "RUN_MODE='full' y WRITE_OUTPUTS=True",
        "created_by_04_03": True,
        "use_after_04_03": "memoria",
    },
    {
        "output_id": "ser_model_comparison_delta.csv",
        "path": relpath(REPORTS_TABLES_DIR / "ser_model_comparison_delta.csv"),
        "write_in_smoke": False,
        "write_in_full": "RUN_MODE='full' y WRITE_OUTPUTS=True",
        "created_by_04_03": True,
        "use_after_04_03": "memoria y comparación metodológica",
    },
    {
        "output_id": "ser_m0_selected_profiles.parquet",
        "path": relpath(MODELING_DIR / "ser_m0_selected_profiles.parquet"),
        "write_in_smoke": False,
        "write_in_full": "RUN_MODE='full' y WRITE_OUTPUTS=True",
        "created_by_04_03": True,
        "use_after_04_03": "modelo M0 seleccionado reutilizable para escenarios y mapas posteriores",
    },
    {
        "output_id": "ser_m0_selected_model_metadata.json",
        "path": relpath(MODELING_DIR / "ser_m0_selected_model_metadata.json"),
        "write_in_smoke": False,
        "write_in_full": "RUN_MODE='full' y WRITE_OUTPUTS=True",
        "created_by_04_03": True,
        "use_after_04_03": "metadatos del modelo M0 seleccionado, trazabilidad y periodo de entrenamiento",
    },
])
display(output_plan)

,run_mode,purpose,representative_results,writes_outputs,expected_use
0,smoke,prueba técnica reducida,False,False,valida estructura
1,full,ejecución completa,True,escritura controlada si WRITE_OUTPUTS=True,resultados interpretables


,output_id,path,write_in_smoke,write_in_full,created_by_04_03,use_after_04_03
0,ser_model_predictions.parquet,data/processed/core/ser/modeling/ser_model_pre...,False,RUN_MODE='full' y WRITE_OUTPUTS=True,True,mapa posterior y agregaciones por barrio/franja
1,ser_model_metrics_comparison.csv,reports/tables/ser_model_metrics_comparison.csv,False,RUN_MODE='full' y WRITE_OUTPUTS=True,True,memoria
2,ser_model_comparison_delta.csv,reports/tables/ser_model_comparison_delta.csv,False,RUN_MODE='full' y WRITE_OUTPUTS=True,True,memoria y comparación metodológica
3,ser_m0_selected_profiles.parquet,data/processed/core/ser/modeling/ser_m0_select...,False,RUN_MODE='full' y WRITE_OUTPUTS=True,True,modelo M0 seleccionado reutilizable para escen...
4,ser_m0_selected_model_metadata.json,data/processed/core/ser/modeling/ser_m0_select...,False,RUN_MODE='full' y WRITE_OUTPUTS=True,True,"metadatos del modelo M0 seleccionado, trazabil..."


## 2. Dataset base de modelado y separación target/features

El dataset de modelado parte del panel final `ser_barrio_intervalo_global_final`. La tabla se transforma en una estructura apta para modelado separando explícitamente:

* claves de identificación;
* target;
* métricas diagnósticas;
* features históricas calculadas sin leakage;
* features ex ante;
* columnas no utilizadas.

La variable objetivo es:

* `ocupacion_pagada_proxy`

Las métricas diagnósticas contemporáneas que no pueden usarse como features del mismo intervalo son:

* `n_tiques_solapados`
* `minutos_solapados`
* `stock_inicio_proxy`
* `stock_fin_proxy`
* `afluencia_tiques`
* `liberacion_tiques`
* `persistencia_completa`
* `saldo_flujo`
* `rotacion_bruta`

Estas columnas pueden utilizarse para revisar el comportamiento del panel y del target, pero no para entrenar modelos predictivos del mismo intervalo.

Las features ex ante iniciales se limitan a variables disponibles antes de `intervalo_inicio`, principalmente variables temporales derivadas de la fecha y hora, codificación del barrio, capacidad anual por barrio y flags temporales simples. En esta versión no se incorporan fuentes adicionales de presión estructural, ya que su integración requiere un tratamiento posterior específico para evitar mezclar la predicción del proxy pagado con componentes no observados de dificultad.

Dado que el panel ya se restringe a ventanas SER observables, no se incorporan flags redundantes como domingo, festivo o día potencialmente observable si no aportan variabilidad real.

In [14]:
panel = pd.read_parquet(PANEL_FINAL_PATH)

panel_period_start = pd.to_datetime(panel["intervalo_inicio"], errors="coerce").min() if "intervalo_inicio" in panel.columns else pd.NaT
panel_period_end = pd.to_datetime(panel["intervalo_inicio"], errors="coerce").max() if "intervalo_inicio" in panel.columns else pd.NaT
granularidad_values = sorted(panel["granularidad_min"].dropna().unique().tolist()) if "granularidad_min" in panel.columns else []

panel_load_summary = pd.DataFrame([
    {
        "dataset_id": "ser_barrio_intervalo_global_final",
        "path": relpath(PANEL_FINAL_PATH),
        "n_rows": len(panel),
        "n_columns": panel.shape[1],
        "memory_mb": round(panel.memory_usage(deep=True).sum() / 1024**2, 3),
        "period_start": panel_period_start,
        "period_end": panel_period_end,
        "n_barrios": panel["barrio_key"].nunique() if "barrio_key" in panel.columns else pd.NA,
        "granularidad_values": granularidad_values,
    }
])
display(panel_load_summary)

required_panel_columns = [
    "barrio_key",
    "intervalo_inicio",
    "intervalo_fin",
    "granularidad_min",
    "ocupacion_pagada_proxy",
]

panel_column_checks_records = []
missing_required_columns = [col for col in required_panel_columns if col not in panel.columns]
panel_column_checks_records.append({
    "check": "required_panel_columns",
    "status": "OK" if not missing_required_columns else "FAIL",
    "detail": "all required columns present" if not missing_required_columns else f"missing: {missing_required_columns}",
})

missing_leakage_columns = [col for col in LEAKAGE_COLUMNS if col not in panel.columns]
panel_column_checks_records.append({
    "check": "diagnostic_leakage_columns_available",
    "status": "OK" if not missing_leakage_columns else "WARNING",
    "detail": "all diagnostic leakage columns present" if not missing_leakage_columns else f"missing diagnostic columns: {missing_leakage_columns}",
})

if missing_required_columns:
    panel_column_checks = pd.DataFrame(panel_column_checks_records)
    display(panel_column_checks)
    raise ValueError(f"Faltan columnas críticas en el panel final SER: {missing_required_columns}")

panel["intervalo_inicio"] = pd.to_datetime(panel["intervalo_inicio"], errors="coerce")
panel["intervalo_fin"] = pd.to_datetime(panel["intervalo_fin"], errors="coerce")
panel = panel.sort_values(["barrio_key", "intervalo_inicio"]).reset_index(drop=True)

invalid_intervalo_inicio = panel["intervalo_inicio"].isna().sum()
invalid_intervalo_fin = panel["intervalo_fin"].isna().sum()
panel_column_checks_records.append({
    "check": "datetime_conversion",
    "status": "OK" if invalid_intervalo_inicio == 0 and invalid_intervalo_fin == 0 else "FAIL",
    "detail": f"invalid intervalo_inicio={invalid_intervalo_inicio}; invalid intervalo_fin={invalid_intervalo_fin}",
})

duplicate_key_count = panel.duplicated(KEY_COLUMNS).sum()
panel_column_checks_records.append({
    "check": "unique_barrio_intervalo_inicio",
    "status": "OK" if duplicate_key_count == 0 else "FAIL",
    "detail": f"duplicated rows by {KEY_COLUMNS}: {duplicate_key_count}",
})

granularity_is_30 = set(panel["granularidad_min"].dropna().unique().tolist()) == {30}
panel_column_checks_records.append({
    "check": "granularidad_30_min",
    "status": "OK" if granularity_is_30 else "WARNING",
    "detail": f"granularidad_min values: {sorted(panel['granularidad_min'].dropna().unique().tolist())}",
})

target_values = pd.to_numeric(panel[TARGET_COLUMN], errors="coerce")
target_missing_count = panel[TARGET_COLUMN].isna().sum()
target_non_finite_count = (~np.isfinite(target_values.to_numpy(dtype=float))).sum() - target_missing_count
target_negative_count = (target_values < 0).sum()
panel_column_checks_records.append({
    "check": "target_basic_validity",
    "status": "OK" if target_missing_count == 0 and target_non_finite_count == 0 and target_negative_count == 0 else "FAIL",
    "detail": f"missing={target_missing_count}; non_finite={target_non_finite_count}; negative={target_negative_count}",
})

panel_column_checks = pd.DataFrame(panel_column_checks_records)
display(panel_column_checks)

failed_checks = panel_column_checks.loc[panel_column_checks["status"].eq("FAIL")]
if not failed_checks.empty:
    raise ValueError(f"Checks críticos fallidos en Sección 2: {failed_checks['check'].tolist()}")

target_finite = target_values[np.isfinite(target_values.to_numpy(dtype=float))]
target_summary = pd.DataFrame([
    {
        "n": len(target_values),
        "n_missing": int(target_missing_count),
        "n_non_finite": int(target_non_finite_count),
        "min": target_finite.min(),
        "p01": target_finite.quantile(0.01),
        "p05": target_finite.quantile(0.05),
        "p50": target_finite.quantile(0.50),
        "p95": target_finite.quantile(0.95),
        "p99": target_finite.quantile(0.99),
        "max": target_finite.max(),
        "mean": target_finite.mean(),
    }
])
display(target_summary)

model_base = panel.copy()
model_base["fecha"] = model_base["intervalo_inicio"].dt.date
model_base["anio"] = model_base["intervalo_inicio"].dt.year
model_base["mes"] = model_base["intervalo_inicio"].dt.month
model_base["dia"] = model_base["intervalo_inicio"].dt.day
model_base["dia_semana_num"] = model_base["intervalo_inicio"].dt.dayofweek
model_base["hora"] = model_base["intervalo_inicio"].dt.hour
model_base["minuto"] = model_base["intervalo_inicio"].dt.minute
model_base["minuto_dia"] = model_base["hora"] * 60 + model_base["minuto"]
model_base["intervalo_30min_id"] = model_base["minuto_dia"] // 30
model_base["sin_hora"] = np.sin(2 * np.pi * model_base["minuto_dia"] / (24 * 60))
model_base["cos_hora"] = np.cos(2 * np.pi * model_base["minuto_dia"] / (24 * 60))
model_base["sin_dia_semana"] = np.sin(2 * np.pi * model_base["dia_semana_num"] / 7)
model_base["cos_dia_semana"] = np.cos(2 * np.pi * model_base["dia_semana_num"] / 7)
model_base["sin_mes"] = np.sin(2 * np.pi * (model_base["mes"] - 1) / 12)
model_base["cos_mes"] = np.cos(2 * np.pi * (model_base["mes"] - 1) / 12)
model_base["es_agosto"] = model_base["mes"].eq(8)
model_base["es_24_31_dic"] = model_base["mes"].eq(12) & model_base["dia"].between(24, 31)

rows_before_smoke = len(model_base)
n_barrios_before_smoke = model_base["barrio_key"].nunique()

if RUN_MODE == "smoke":
    SMOKE_START = pd.Timestamp("2023-01-02")
    SMOKE_END = pd.Timestamp("2023-02-28 23:59:59")
    smoke_barrios = model_base["barrio_key"].value_counts().head(8).index.tolist()
    smoke_mask = model_base["barrio_key"].isin(smoke_barrios) & model_base["intervalo_inicio"].between(SMOKE_START, SMOKE_END)
    model_base = model_base.loc[smoke_mask].copy().reset_index(drop=True)
    smoke_note = "reducción técnica no representativa"
else:
    smoke_note = "sin reducción aplicada"

smoke_filter_summary = pd.DataFrame([
    {
        "run_mode": RUN_MODE,
        "rows_before": rows_before_smoke,
        "rows_after": len(model_base),
        "n_barrios_before": n_barrios_before_smoke,
        "n_barrios_after": model_base["barrio_key"].nunique(),
        "period_start_after": model_base["intervalo_inicio"].min(),
        "period_end_after": model_base["intervalo_inicio"].max(),
        "note": smoke_note,
    }
])
display(smoke_filter_summary)

available_leakage_columns = [col for col in LEAKAGE_COLUMNS if col in model_base.columns]
available_forbidden_feature_columns = [col for col in FORBIDDEN_FEATURE_COLUMNS if col in model_base.columns]
key_columns_available = [col for col in KEY_COLUMNS if col in model_base.columns]
target_columns = [TARGET_COLUMN] if TARGET_COLUMN in model_base.columns else []
diagnostic_columns_available = available_leakage_columns.copy()

candidate_ex_ante_columns = [
    "barrio_key",
    "anio",
    "mes",
    "dia",
    "dia_semana_num",
    "hora",
    "minuto",
    "minuto_dia",
    "intervalo_30min_id",
    "sin_hora",
    "cos_hora",
    "sin_dia_semana",
    "cos_dia_semana",
    "sin_mes",
    "cos_mes",
    "es_agosto",
    "es_24_31_dic",
    "plazas_barrio_anio",
    "n_calles_barrio_anio",
    "cod_distrito",
    "cod_barrio",
]
ex_ante_feature_candidates = [
    col for col in candidate_ex_ante_columns
    if col in model_base.columns and col not in available_forbidden_feature_columns
]


def compact_columns(columns: list[str], max_items: int = 20) -> list[str]:
    return columns[:max_items] + ([f"... (+{len(columns) - max_items})"] if len(columns) > max_items else [])


column_role_summary = pd.DataFrame([
    {"role": "keys", "n_columns": len(key_columns_available), "columns": compact_columns(key_columns_available)},
    {"role": "target", "n_columns": len(target_columns), "columns": compact_columns(target_columns)},
    {"role": "diagnostic_not_features", "n_columns": len(diagnostic_columns_available), "columns": compact_columns(diagnostic_columns_available)},
    {"role": "forbidden_features", "n_columns": len(available_forbidden_feature_columns), "columns": compact_columns(available_forbidden_feature_columns)},
    {"role": "ex_ante_feature_candidates", "n_columns": len(ex_ante_feature_candidates), "columns": compact_columns(ex_ante_feature_candidates)},
])
display(column_role_summary)

model_base_summary = pd.DataFrame([
    {
        "run_mode": RUN_MODE,
        "n_rows": len(model_base),
        "n_columns": model_base.shape[1],
        "n_barrios": model_base["barrio_key"].nunique(),
        "period_start": model_base["intervalo_inicio"].min(),
        "period_end": model_base["intervalo_inicio"].max(),
        "target": TARGET_COLUMN,
        "n_ex_ante_feature_candidates": len(ex_ante_feature_candidates),
        "n_forbidden_feature_columns_available": len(available_forbidden_feature_columns),
    }
])
display(model_base_summary)

,dataset_id,path,n_rows,n_columns,memory_mb,period_start,period_end,n_barrios,granularidad_values
0,ser_barrio_intervalo_global_final,data/processed/core/ser/ser_barrio_intervalo_g...,1292148,27,280.962,2023-01-02 09:00:00,2026-03-31 20:30:00,65,[30]


,check,status,detail
0,required_panel_columns,OK,all required columns present
1,diagnostic_leakage_columns_available,OK,all diagnostic leakage columns present
2,datetime_conversion,OK,invalid intervalo_inicio=0; invalid intervalo_...
3,unique_barrio_intervalo_inicio,OK,"duplicated rows by ['barrio_key', 'intervalo_i..."
4,granularidad_30_min,OK,granularidad_min values: [30]
5,target_basic_validity,OK,missing=0; non_finite=0; negative=0


,n,n_missing,n_non_finite,min,p01,p05,p50,p95,p99,max,mean
0,1292148,0,0,0.0,0.0,0.021869,0.089994,0.175232,0.215672,0.325619,0.094575


,run_mode,rows_before,rows_after,n_barrios_before,n_barrios_after,period_start_after,period_end_after,note
0,full,1292148,1292148,65,65,2023-01-02 09:00:00,2026-03-31 20:30:00,sin reducción aplicada


,role,n_columns,columns
0,keys,2,"[barrio_key, intervalo_inicio]"
1,target,1,[ocupacion_pagada_proxy]
2,diagnostic_not_features,9,"[n_tiques_solapados, minutos_solapados, stock_..."
3,forbidden_features,10,"[ocupacion_pagada_proxy, n_tiques_solapados, m..."
4,ex_ante_feature_candidates,18,"[barrio_key, anio, mes, dia, dia_semana_num, h..."


,run_mode,n_rows,n_columns,n_barrios,period_start,period_end,target,n_ex_ante_feature_candidates,n_forbidden_feature_columns_available
0,full,1292148,43,65,2023-01-02 09:00:00,2026-03-31 20:30:00,ocupacion_pagada_proxy,18,10


La carga del panel final confirma que el dataset de modelado se construye sobre la salida completa de `ser_barrio_intervalo_global_final`, sin reducción muestral en modo `full`. La tabla contiene 1.292.148 observaciones, 65 barrios y una única granularidad temporal de 30 minutos, con cobertura efectiva desde el 2 de enero de 2023 hasta el 31 de marzo de 2026. Por tanto, el notebook trabaja con la unidad analítica prevista: `barrio_key × intervalo_inicio`.

Los checks básicos del panel son satisfactorios: están presentes las columnas requeridas, las fechas se convierten correctamente, no aparecen duplicados por `barrio_key × intervalo_inicio`, la granularidad es homogénea y el target no presenta valores nulos, infinitos ni negativos. Esto permite continuar con la construcción de splits y modelos sin aplicar filtros adicionales sobre el panel base.

La distribución de `ocupacion_pagada_proxy` debe interpretarse con cautela. La media del proxy es 0,0946, la mediana 0,0900, el percentil 95 es 0,1752 y el valor máximo es 0,3256. Estos valores no representan ocupación real de plazas SER, sino presión pagada observada respecto a la capacidad-tiempo total del barrio en cada intervalo. La escala relativamente baja es coherente con la definición del proxy y con la unidad espacial adoptada: el panel agrega los minutos pagados solapados de los tiques asignados al barrio y los divide entre todas las plazas SER disponibles en ese barrio durante el intervalo. Por tanto, pueden existir calles, tramos o zonas concretas con presión muy elevada aunque el valor medio agregado del barrio sea moderado. Además, el numerador solo recoge señal pagada observada, mientras que no incorpora residentes, vehículos exentos, ocupación irregular ni demanda no materializada en tique. En consecuencia, los intervalos sin tiques o con bajo valor del proxy no deben interpretarse automáticamente como facilidad real para aparcar.

La separación entre columnas de identificación, target, métricas diagnósticas y features candidatas queda correctamente definida. Las métricas contemporáneas derivadas de los tiques del propio intervalo se conservan para diagnóstico, pero se excluyen como predictores para evitar leakage. El conjunto inicial de features ex ante queda limitado a variables temporales, identificadores espaciales y capacidad estructural disponible antes del intervalo objetivo. Esta separación es metodológicamente necesaria porque el objetivo del notebook es evaluar predicción ex ante del proxy histórico, no reconstrucción retrospectiva usando información del mismo intervalo.


## 3. Diseño de splits temporales

La validación del notebook es temporal. No se utilizará validación cruzada aleatoria, ya que mezclaría información futura y pasada de forma incompatible con el objetivo operativo del modelo.

Se definen dos evaluaciones:

1. `split_validation`

   * entrenamiento: 2023-2024;
   * validación: 2025.

2. `split_test_refit`

   * entrenamiento: 2023-2025;
   * test: 2026 Q1.

Las fechas anteriores definen las ventanas calendario de cada split, pero la cobertura efectiva depende de los intervalos realmente presentes en el panel. Por ello, las tablas de comprobación muestran tanto la definición teórica como el periodo observado efectivo de entrenamiento y evaluación. Esta distinción evita interpretar como ausencia de datos un simple ajuste derivado del calendario SER observable y de la cobertura real del panel.

La primera evaluación sirve para comparar modelos y revisar si las variables ex ante mejoran al baseline histórico. La segunda evaluación estima el rendimiento final sobre 2026 Q1 utilizando todo el histórico disponible antes de 2026.

En modo `smoke`, se utilizará una ventana temporal y un conjunto reducido de barrios para validar la arquitectura sin esperar a la ejecución completa. Los resultados de este modo no se interpretan como rendimiento del modelo.

Además de construir los índices de entrenamiento y evaluación, la celda valida internamente que cada split tenga filas en ambos tramos, que no exista solapamiento entre entrenamiento y evaluación, que el periodo de entrenamiento sea anterior al periodo evaluado y que la cobertura de barrios de evaluación frente al entrenamiento quede documentada. La presencia de barrios no vistos en train se trata como advertencia metodológica, no como error bloqueante, porque el baseline dispone de niveles de fallback.

In [15]:
if RUN_MODE == "full":
    split_definitions = [
        {
            "split_id": "split_validation",
            "train_start": pd.Timestamp("2023-01-01 00:00:00"),
            "train_end": pd.Timestamp("2024-12-31 23:59:59"),
            "eval_start": pd.Timestamp("2025-01-01 00:00:00"),
            "eval_end": pd.Timestamp("2025-12-31 23:59:59"),
            "eval_label": "validation",
        },
        {
            "split_id": "split_test_refit",
            "train_start": pd.Timestamp("2023-01-01 00:00:00"),
            "train_end": pd.Timestamp("2025-12-31 23:59:59"),
            "eval_start": pd.Timestamp("2026-01-01 00:00:00"),
            "eval_end": pd.Timestamp("2026-03-31 23:59:59"),
            "eval_label": "test",
        },
    ]
elif RUN_MODE == "smoke":
    split_definitions = [
        {
            "split_id": "split_smoke_validation",
            "train_start": pd.Timestamp("2023-01-02 00:00:00"),
            "train_end": pd.Timestamp("2023-01-31 23:59:59"),
            "eval_start": pd.Timestamp("2023-02-01 00:00:00"),
            "eval_end": pd.Timestamp("2023-02-14 23:59:59"),
            "eval_label": "validation_smoke",
        },
        {
            "split_id": "split_smoke_test_refit",
            "train_start": pd.Timestamp("2023-01-02 00:00:00"),
            "train_end": pd.Timestamp("2023-02-14 23:59:59"),
            "eval_start": pd.Timestamp("2023-02-15 00:00:00"),
            "eval_end": pd.Timestamp("2023-02-28 23:59:59"),
            "eval_label": "test_smoke",
        },
    ]
else:
    raise ValueError(f"RUN_MODE no válido: {RUN_MODE!r}")


def build_split_indices(df: pd.DataFrame, split_definitions: list[dict]) -> dict:
    split_indices = {}
    intervalo_inicio = pd.to_datetime(df["intervalo_inicio"], errors="coerce")
    for definition in split_definitions:
        train_mask = intervalo_inicio.between(definition["train_start"], definition["train_end"], inclusive="both")
        eval_mask = intervalo_inicio.between(definition["eval_start"], definition["eval_end"], inclusive="both")
        split_indices[definition["split_id"]] = {
            "train_idx": df.index[train_mask].copy(),
            "eval_idx": df.index[eval_mask].copy(),
            "definition": definition.copy(),
        }
    return split_indices


split_indices = build_split_indices(model_base, split_definitions)

split_check_records = []
split_summary_records = []
split_barrio_coverage_records = []
split_barrio_coverage_detail_records = []
temporal_leakage_records = []

for definition in split_definitions:
    split_id = definition["split_id"]
    train_idx = split_indices[split_id]["train_idx"]
    eval_idx = split_indices[split_id]["eval_idx"]
    train_df = model_base.loc[train_idx]
    eval_df = model_base.loc[eval_idx]

    train_empty = train_df.empty
    eval_empty = eval_df.empty
    train_eval_overlap = len(train_idx.intersection(eval_idx))
    train_max_intervalo_inicio = train_df["intervalo_inicio"].max() if not train_empty else pd.NaT
    eval_min_intervalo_inicio = eval_df["intervalo_inicio"].min() if not eval_empty else pd.NaT
    train_before_eval_definition = definition["train_end"] < definition["eval_start"]
    train_before_eval_actual = bool(train_max_intervalo_inicio < eval_min_intervalo_inicio) if not train_empty and not eval_empty else False

    split_check_records.extend([
        {
            "split_id": split_id,
            "check": "train_not_empty",
            "status": "OK" if not train_empty else "FAIL",
            "detail": f"n_train_rows={len(train_df)}",
        },
        {
            "split_id": split_id,
            "check": "eval_not_empty",
            "status": "OK" if not eval_empty else "FAIL",
            "detail": f"n_eval_rows={len(eval_df)}",
        },
        {
            "split_id": split_id,
            "check": "definition_train_end_before_eval_start",
            "status": "OK" if train_before_eval_definition else "FAIL",
            "detail": f"train_end={definition['train_end']}; eval_start={definition['eval_start']}",
        },
        {
            "split_id": split_id,
            "check": "no_row_overlap_train_eval",
            "status": "OK" if train_eval_overlap == 0 else "FAIL",
            "detail": f"overlap_rows={train_eval_overlap}",
        },
        {
            "split_id": split_id,
            "check": "actual_train_max_before_eval_min",
            "status": "OK" if train_before_eval_actual else "FAIL",
            "detail": f"train_max={train_max_intervalo_inicio}; eval_min={eval_min_intervalo_inicio}",
        },
    ])

    train_barrios = set(train_df["barrio_key"].dropna().unique())
    eval_barrios = set(eval_df["barrio_key"].dropna().unique())
    common_barrios = sorted(train_barrios & eval_barrios)
    eval_only_barrios = sorted(eval_barrios - train_barrios)
    split_check_records.append({
        "split_id": split_id,
        "check": "eval_barrios_covered_by_train",
        "status": "OK" if len(eval_only_barrios) == 0 else "WARNING",
        "detail": f"n_eval_only_barrios={len(eval_only_barrios)}; preview={eval_only_barrios[:10]}",
    })

    split_summary_records.append({
        "run_mode": RUN_MODE,
        "split_id": split_id,
        "eval_label": definition["eval_label"],
        "train_start_definition": definition["train_start"],
        "train_end_definition": definition["train_end"],
        "eval_start_definition": definition["eval_start"],
        "eval_end_definition": definition["eval_end"],
        "n_train_rows": len(train_df),
        "n_eval_rows": len(eval_df),
        "n_train_barrios": len(train_barrios),
        "n_eval_barrios": len(eval_barrios),
        "train_period_start_actual": train_df["intervalo_inicio"].min() if not train_empty else pd.NaT,
        "train_period_end_actual": train_max_intervalo_inicio,
        "eval_period_start_actual": eval_min_intervalo_inicio,
        "eval_period_end_actual": eval_df["intervalo_inicio"].max() if not eval_empty else pd.NaT,
        "n_common_barrios_train_eval": len(common_barrios),
        "n_eval_barrios_not_in_train": len(eval_only_barrios),
    })

    split_barrio_coverage_records.append({
        "split_id": split_id,
        "n_train_barrios": len(train_barrios),
        "n_eval_barrios": len(eval_barrios),
        "n_common_barrios": len(common_barrios),
        "n_eval_only_barrios": len(eval_only_barrios),
        "eval_only_barrios_preview": eval_only_barrios[:10],
    })

    for barrio_key in sorted(train_barrios | eval_barrios):
        split_barrio_coverage_detail_records.append({
            "split_id": split_id,
            "barrio_key": barrio_key,
            "in_train": barrio_key in train_barrios,
            "in_eval": barrio_key in eval_barrios,
        })

    temporal_leakage_records.append({
        "split_id": split_id,
        "train_max_intervalo_inicio": train_max_intervalo_inicio,
        "eval_min_intervalo_inicio": eval_min_intervalo_inicio,
        "train_before_eval": train_before_eval_actual,
        "status": "OK" if train_before_eval_actual else "FAIL",
    })

split_checks = pd.DataFrame(split_check_records)
split_summary = pd.DataFrame(split_summary_records)
split_barrio_coverage_summary = pd.DataFrame(split_barrio_coverage_records)
split_barrio_coverage_detail = pd.DataFrame(split_barrio_coverage_detail_records)
temporal_leakage_split_checks = pd.DataFrame(temporal_leakage_records)

split_status_records = []
for split_id in split_summary["split_id"]:
    split_statuses = split_checks.loc[split_checks["split_id"].eq(split_id), "status"].tolist()
    temporal_statuses = temporal_leakage_split_checks.loc[temporal_leakage_split_checks["split_id"].eq(split_id), "status"].tolist()
    statuses = split_statuses + temporal_statuses
    checks_status = "FAIL" if "FAIL" in statuses else "WARNING" if "WARNING" in statuses else "OK"
    split_status_records.append({"split_id": split_id, "checks_status": checks_status})

split_status_table = pd.DataFrame(split_status_records)
split_summary_compact = (
    split_summary.assign(
        train_period_actual=lambda df: df["train_period_start_actual"].astype(str) + " → " + df["train_period_end_actual"].astype(str),
        eval_period_actual=lambda df: df["eval_period_start_actual"].astype(str) + " → " + df["eval_period_end_actual"].astype(str),
    )
    .merge(split_status_table, on="split_id", how="left")
    [[
        "run_mode",
        "split_id",
        "eval_label",
        "train_period_actual",
        "eval_period_actual",
        "n_train_rows",
        "n_eval_rows",
        "n_train_barrios",
        "n_eval_barrios",
        "n_eval_barrios_not_in_train",
        "checks_status",
    ]]
)

display(split_summary_compact)

split_checks_with_issues = split_checks.loc[split_checks["status"].isin(["FAIL", "WARNING"])]
failed_temporal_leakage_checks = temporal_leakage_split_checks.loc[temporal_leakage_split_checks["status"].eq("FAIL")]
temporal_leakage_checks_with_issues = temporal_leakage_split_checks.loc[temporal_leakage_split_checks["status"].isin(["FAIL", "WARNING"])]
has_split_issues = not split_checks_with_issues.empty
has_temporal_issues = not temporal_leakage_checks_with_issues.empty

if SHOW_DEBUG_TABLES or has_split_issues:
    display(split_checks)
if SHOW_DEBUG_TABLES:
    display(split_barrio_coverage_summary)
    display(split_barrio_coverage_detail)
if SHOW_DEBUG_TABLES or has_temporal_issues:
    display(temporal_leakage_split_checks)

failed_split_checks = split_checks.loc[split_checks["status"].eq("FAIL")]
if not failed_split_checks.empty:
    raise ValueError(f"Checks de splits fallidos: {failed_split_checks[['split_id', 'check']].to_dict('records')}")
if not failed_temporal_leakage_checks.empty:
    raise ValueError(f"Checks de leakage temporal fallidos: {failed_temporal_leakage_checks['split_id'].tolist()}")

,run_mode,split_id,eval_label,train_period_actual,eval_period_actual,n_train_rows,n_eval_rows,n_train_barrios,n_eval_barrios,n_eval_barrios_not_in_train,checks_status
0,full,split_validation,validation,2023-01-02 09:00:00 → 2024-12-31 14:30:00,2025-01-02 09:00:00 → 2025-12-31 14:30:00,775008,410280,63,65,2,WARNING
1,full,split_test_refit,test,2023-01-02 09:00:00 → 2025-12-31 14:30:00,2026-01-02 09:00:00 → 2026-03-31 20:30:00,1185288,106860,65,65,0,OK


,split_id,check,status,detail
0,split_validation,train_not_empty,OK,n_train_rows=775008
1,split_validation,eval_not_empty,OK,n_eval_rows=410280
2,split_validation,definition_train_end_before_eval_start,OK,train_end=2024-12-31 23:59:59; eval_start=2025...
3,split_validation,no_row_overlap_train_eval,OK,overlap_rows=0
4,split_validation,actual_train_max_before_eval_min,OK,train_max=2024-12-31 14:30:00; eval_min=2025-0...
5,split_validation,eval_barrios_covered_by_train,WARNING,"n_eval_only_barrios=2; preview=['11_01', '12_06']"
6,split_test_refit,train_not_empty,OK,n_train_rows=1185288
7,split_test_refit,eval_not_empty,OK,n_eval_rows=106860
8,split_test_refit,definition_train_end_before_eval_start,OK,train_end=2025-12-31 23:59:59; eval_start=2026...
9,split_test_refit,no_row_overlap_train_eval,OK,overlap_rows=0


La validación se estructura mediante dos particiones temporales coherentes con el objetivo predictivo del TFM. El primer split, `split_validation`, entrena con observaciones de 2023-2024 y evalúa sobre 2025. Este split actúa como partición principal de comparación metodológica entre el baseline histórico y las variantes del modelo mínimo. El segundo split, `split_test_refit`, reentrena con 2023-2025 y evalúa sobre 2026 Q1, utilizando todo el histórico disponible antes del periodo final de prueba.

Los checks temporales son satisfactorios: ambos splits tienen filas de entrenamiento y evaluación, no existe solapamiento entre train y evaluación, y el máximo temporal del entrenamiento es anterior al mínimo temporal de evaluación. Por tanto, la evaluación respeta el orden cronológico y evita mezclar información futura con información pasada.

La única advertencia aparece en `split_validation`: dos barrios presentes en 2025 no aparecen en el tramo de entrenamiento 2023-2024. Esta situación no bloquea el notebook porque responde a la ampliación temporal de la cobertura SER y porque el baseline dispone de niveles de fallback para producir predicciones cuando no existe histórico específico del barrio. No obstante, debe interpretarse como una limitación metodológica: en esos barrios, la predicción de validación no se apoya en un perfil histórico propio de barrio, sino en patrones más generales.

En `split_test_refit` no aparece esta limitación: el entrenamiento 2023-2025 cubre los 65 barrios evaluados en 2026 Q1. Por tanto, el split final es más estable desde el punto de vista espacial y permite evaluar el rendimiento del modelo seleccionado sobre el primer trimestre de 2026 sin barrios completamente nuevos en test.

La estrategia de splits es adecuada para el problema porque evalúa la capacidad de generalización temporal del modelo y evita una validación cruzada aleatoria, que mezclaría intervalos cercanos en el tiempo y podría producir una estimación demasiado optimista del rendimiento.


## 4. Estrategia de modelado y selección de estimadores

El problema se plantea como una tarea de regresión supervisada espacio-temporal sobre un panel `barrio_key × intervalo_inicio`. Aunque las observaciones están ordenadas en el tiempo, no se trata de una única serie temporal, sino de múltiples trayectorias por barrio e intervalo de 30 minutos. Por ello, la validación se diseña temporalmente, pero el enfoque de modelado se formula como aprendizaje supervisado sobre una tabla panel.

La primera referencia es un baseline histórico no paramétrico. Este baseline estima el target mediante perfiles medios calculados exclusivamente con el tramo de entrenamiento de cada split, usando una jerarquía de agregaciones por barrio, día de semana e intervalo horario. Su función no es maximizar complejidad predictiva, sino establecer una referencia mínima trazable: si un modelo posterior no mejora este patrón histórico, no aporta evidencia suficiente de valor añadido.

Sobre esa referencia se plantea un modelo mínimo con variables disponibles ex ante. En esta versión del notebook se utiliza Ridge como estimador lineal regularizado, porque ofrece una relación adecuada entre simplicidad, reproducibilidad, robustez frente a colinealidad y coste computacional. Además, permite evaluar si las variables temporales, espaciales y estructurales añaden señal sobre el perfil histórico sin introducir una complejidad difícil de justificar en esta fase del TFM.

El uso de modelos clásicos de series temporales no se adopta como enfoque principal porque el dataset no corresponde a una serie univariante, sino a un panel espacio-temporal con múltiples barrios, calendarios observables SER y variables explicativas heterogéneas. Una familia ARIMA/SARIMA por barrio exigiría entrenar y mantener múltiples modelos, dificultaría la comparación homogénea entre unidades espaciales y no incorporaría de forma directa variables estructurales o espaciales. No obstante, la dimensión temporal sí se incorpora mediante splits cronológicos, variables cíclicas, perfiles históricos y controles explícitos de leakage.

De forma condicionada, podría probarse un modelo no lineal ligero si el modelo mínimo regularizado no mejora suficientemente al baseline o si los errores muestran patrones no lineales claros. Esta extensión solo tendría sentido si mejora de forma consistente las métricas temporales sin desplazar el objetivo principal del notebook: construir una comparación reproducible, interpretable y metodológicamente defendible entre una referencia histórica y un modelo con información ex ante.


## 5. Baseline histórico

El baseline histórico representa la referencia mínima del TFM. Su objetivo es estimar `ocupacion_pagada_proxy` usando únicamente patrones aprendidos del histórico pasado del propio panel, sin fuentes externas ni métricas contemporáneas del intervalo objetivo.

Para cada split temporal, el baseline se ajusta exclusivamente con el tramo de entrenamiento correspondiente y se aplica después sobre el tramo de evaluación. De este modo, para predecir 2025 solo se utilizan perfiles aprendidos con 2023-2024, y para predecir 2026 Q1 se utilizan perfiles aprendidos con 2023-2025.

La lógica del baseline es una jerarquía de medias históricas con fallback:

1. media por `barrio_key × dia_semana_num × intervalo_30min_id`;
2. media por `barrio_key × intervalo_30min_id`;
3. media por `barrio_key`;
4. media por `dia_semana_num × intervalo_30min_id`;
5. media global del entrenamiento.

Esta jerarquía permite producir predicciones incluso cuando una combinación concreta de barrio, día de semana e intervalo horario no aparece en el entrenamiento. El nivel utilizado en cada predicción queda registrado en `fallback_level`, lo que permite comprobar si el baseline está prediciendo con perfiles específicos de barrio o si depende de niveles más generales.

El baseline no debe calcularse con todo el periodo 2023-2026 antes de separar train, validación y test. Hacerlo introduciría leakage retrospectivo, ya que los perfiles históricos incorporarían información del propio periodo evaluado.

La sección genera tres objetos principales: `baseline_predictions`, con predicciones y errores por barrio e intervalo; `baseline_summary`, con métricas agregadas por split; y `baseline_fallback_summary`, con el reparto de predicciones por nivel de fallback. Además, se ejecutan checks para verificar que existen filas de entrenamiento y evaluación, que el target no contiene nulos, que las predicciones son finitas y que las claves usadas por el baseline no incluyen columnas prohibidas.

In [16]:
class HistoricalProfileBaseline:
    def __init__(self) -> None:
        self.hierarchy = [
            ("barrio_dow_interval", ["barrio_key", "dia_semana_num", "intervalo_30min_id"]),
            ("barrio_interval", ["barrio_key", "intervalo_30min_id"]),
            ("barrio", ["barrio_key"]),
            ("dow_interval", ["dia_semana_num", "intervalo_30min_id"]),
        ]
        self.profiles_ = {}
        self.global_mean_ = np.nan
        self.target_col_ = None

    def fit(self, train_df: pd.DataFrame, target_col: str) -> "HistoricalProfileBaseline":
        self.target_col_ = target_col
        train = train_df.copy()
        train[target_col] = pd.to_numeric(train[target_col], errors="coerce")
        finite_train = train.loc[np.isfinite(train[target_col].to_numpy(dtype=float))]
        self.global_mean_ = finite_train[target_col].mean()
        self.profiles_ = {
            level_name: finite_train.groupby(level_columns, dropna=False)[target_col].mean()
            for level_name, level_columns in self.hierarchy
        }
        return self

    def predict(self, eval_df: pd.DataFrame) -> pd.DataFrame:
        y_pred = pd.Series(np.nan, index=eval_df.index, dtype="float64")
        fallback_level = pd.Series(pd.NA, index=eval_df.index, dtype="object")

        for level_name, level_columns in self.hierarchy:
            profile = self.profiles_[level_name].reset_index(name="_profile_prediction")
            matched = eval_df[level_columns].reset_index().merge(profile, on=level_columns, how="left").set_index("index")
            level_predictions = matched["_profile_prediction"].reindex(eval_df.index)
            fill_mask = y_pred.isna() & level_predictions.notna()
            y_pred.loc[fill_mask] = level_predictions.loc[fill_mask]
            fallback_level.loc[fill_mask] = level_name

        fill_global_mask = y_pred.isna()
        y_pred.loc[fill_global_mask] = self.global_mean_
        fallback_level.loc[fill_global_mask] = "global_mean"

        return pd.DataFrame({"y_pred": y_pred, "fallback_level": fallback_level}, index=eval_df.index)


model_id = "M0_historical_profile"
baseline_prediction_frames = []
baseline_check_records = []
baseline_summary_records = []
baseline_feature_columns = ["barrio_key", "dia_semana_num", "intervalo_30min_id"]
forbidden_predictor_overlap = sorted(set(baseline_feature_columns) & set(available_forbidden_feature_columns))

for split_id, split_data in split_indices.items():
    definition = split_data["definition"]
    train_df = model_base.loc[split_data["train_idx"]].copy()
    eval_df = model_base.loc[split_data["eval_idx"]].copy()
    train_target = pd.to_numeric(train_df[TARGET_COLUMN], errors="coerce")
    eval_target = pd.to_numeric(eval_df[TARGET_COLUMN], errors="coerce")

    train_empty = train_df.empty
    eval_empty = eval_df.empty
    train_target_missing = train_target.isna().sum()
    eval_target_missing = eval_target.isna().sum()

    baseline = HistoricalProfileBaseline().fit(train_df, TARGET_COLUMN)
    predictions = baseline.predict(eval_df)
    y_pred = pd.to_numeric(predictions["y_pred"], errors="coerce")
    y_true = eval_target.reindex(eval_df.index)
    prediction_errors = y_pred - y_true

    split_predictions = pd.DataFrame({
        "model_id": model_id,
        "split_id": split_id,
        "eval_label": definition["eval_label"],
        "barrio_key": eval_df["barrio_key"],
        "intervalo_inicio": eval_df["intervalo_inicio"],
        "y_true": y_true,
        "y_pred": y_pred,
        "error": prediction_errors,
        "abs_error": prediction_errors.abs(),
        "fallback_level": predictions["fallback_level"],
    })
    baseline_prediction_frames.append(split_predictions)

    pred_missing = y_pred.isna().sum()
    pred_non_finite = (~np.isfinite(y_pred.to_numpy(dtype=float))).sum() - pred_missing
    fallback_missing = predictions["fallback_level"].isna().sum()
    global_mean_finite = bool(np.isfinite(baseline.global_mean_))

    baseline_check_records.extend([
        {"split_id": split_id, "check": "train_not_empty", "status": "OK" if not train_empty else "FAIL", "detail": f"n_train_rows={len(train_df)}"},
        {"split_id": split_id, "check": "eval_not_empty", "status": "OK" if not eval_empty else "FAIL", "detail": f"n_eval_rows={len(eval_df)}"},
        {"split_id": split_id, "check": "train_target_not_null", "status": "OK" if train_target_missing == 0 else "FAIL", "detail": f"missing_train_target={train_target_missing}"},
        {"split_id": split_id, "check": "eval_target_not_null", "status": "OK" if eval_target_missing == 0 else "FAIL", "detail": f"missing_eval_target={eval_target_missing}"},
        {"split_id": split_id, "check": "global_mean_finite", "status": "OK" if global_mean_finite else "FAIL", "detail": f"global_mean={baseline.global_mean_}"},
        {"split_id": split_id, "check": "predictions_not_null", "status": "OK" if pred_missing == 0 else "FAIL", "detail": f"missing_predictions={pred_missing}"},
        {"split_id": split_id, "check": "predictions_finite", "status": "OK" if pred_non_finite == 0 else "FAIL", "detail": f"non_finite_predictions={pred_non_finite}"},
        {"split_id": split_id, "check": "n_predictions_matches_eval", "status": "OK" if len(split_predictions) == len(eval_df) else "FAIL", "detail": f"n_predictions={len(split_predictions)}; n_eval_rows={len(eval_df)}"},
        {"split_id": split_id, "check": "fallback_level_not_null", "status": "OK" if fallback_missing == 0 else "FAIL", "detail": f"missing_fallback_level={fallback_missing}"},
        {"split_id": split_id, "check": "no_forbidden_explicit_predictors", "status": "OK" if not forbidden_predictor_overlap else "FAIL", "detail": f"forbidden_predictor_overlap={forbidden_predictor_overlap}"},
    ])

    baseline_summary_records.append({
        "model_id": model_id,
        "split_id": split_id,
        "eval_label": definition["eval_label"],
        "n_train_rows": len(train_df),
        "n_eval_rows": len(eval_df),
        "n_predictions": len(split_predictions),
        "train_target_mean": train_target.mean(),
        "eval_target_mean": y_true.mean(),
        "pred_mean": y_pred.mean(),
        "mae": split_predictions["abs_error"].mean(),
        "rmse": np.sqrt(np.mean(np.square(split_predictions["error"]))),
        "bias": prediction_errors.mean(),
        "fallback_global_pct": split_predictions["fallback_level"].eq("global_mean").mean() * 100,
    })

baseline_predictions = pd.concat(baseline_prediction_frames, ignore_index=True)
baseline_checks = pd.DataFrame(baseline_check_records)

baseline_status = (
    baseline_checks.assign(status_rank=lambda df: df["status"].map({"OK": 0, "WARNING": 1, "FAIL": 2}))
    .groupby("split_id", as_index=False)["status_rank"]
    .max()
    .assign(checks_status=lambda df: df["status_rank"].map({0: "OK", 1: "WARNING", 2: "FAIL"}))
    [["split_id", "checks_status"]]
)
baseline_summary = pd.DataFrame(baseline_summary_records).merge(baseline_status, on="split_id", how="left")
baseline_fallback_summary = (
    baseline_predictions.groupby(["split_id", "fallback_level"], dropna=False)
    .size()
    .reset_index(name="n_predictions")
)
baseline_fallback_summary["pct_predictions"] = baseline_fallback_summary.groupby("split_id")["n_predictions"].transform(lambda values: values / values.sum() * 100)

display(baseline_summary)
display(baseline_fallback_summary)

baseline_checks_with_issues = baseline_checks.loc[baseline_checks["status"].isin(["WARNING", "FAIL"])]
if SHOW_DEBUG_TABLES or not baseline_checks_with_issues.empty:
    display(baseline_checks)

failed_baseline_checks = baseline_checks.loc[baseline_checks["status"].eq("FAIL")]
if not failed_baseline_checks.empty:
    raise ValueError(f"Checks de baseline fallidos: {failed_baseline_checks[['split_id', 'check']].to_dict('records')}")

,model_id,split_id,eval_label,n_train_rows,n_eval_rows,n_predictions,train_target_mean,eval_target_mean,pred_mean,mae,rmse,bias,fallback_global_pct,checks_status
0,M0_historical_profile,split_validation,validation,775008,410280,410280,0.092472,0.097507,0.090598,0.017702,0.026926,-0.006910,0.0,OK
1,M0_historical_profile,split_test_refit,test,1185288,106860,106860,0.094215,0.098564,0.091749,0.013168,0.017864,-0.006815,0.0,OK


,split_id,fallback_level,n_predictions,pct_predictions
0,split_test_refit,barrio_dow_interval,106860,100.000000
1,split_validation,barrio_dow_interval,397656,96.923077
2,split_validation,dow_interval,12624,3.076923


En `split_validation`, el baseline entrena con 775.008 observaciones y evalúa sobre 410.280 intervalos de 2025. La media del target en entrenamiento es 0,0925, mientras que la media observada en validación asciende a 0,0975. La media predicha por M0 es 0,0906, lo que genera un sesgo negativo de -0,0069. Este resultado indica que el baseline tiende a infraestimar la presión pagada media de 2025, aunque mantiene un error absoluto medio moderado respecto a la escala del proxy.

En `split_test_refit`, el baseline entrena con 1.185.288 observaciones y evalúa sobre 106.860 intervalos de 2026 Q1. La media observada del target en test es 0,0986 y la media predicha es 0,0917, con un sesgo negativo similar al observado en validación. El MAE desciende de 0,0177 en validación a 0,0132 en test, por lo que el rendimiento final sobre 2026 Q1 resulta más favorable que el observado en 2025.

La tabla de fallback muestra que en `split_test_refit` el 100% de las predicciones se generan con el nivel más específico, `barrio_key × dia_semana_num × intervalo_30min_id`. En `split_validation`, el 96,92% también utiliza ese nivel específico, mientras que el 3,08% recurre al fallback `dia_semana_num × intervalo_30min_id`. Esta proporción coincide con la presencia de barrios nuevos en validación y confirma que la advertencia de la Sección 3 afecta a una fracción limitada de las predicciones.

En conjunto, M0 constituye una referencia fuerte e interpretable: captura patrones históricos por barrio, día de semana e intervalo horario, no depende de información contemporánea del periodo evaluado y produce predicciones completas. Su principal limitación inicial es la tendencia a infraestimar la media del target en los periodos de evaluación, lo que justifica contrastarlo con variantes que incorporen una recalibración o features ex ante.


## 6. Modelo mínimo histórico + features ex ante

El modelo mínimo no sustituye al baseline histórico: lo extiende. Su propósito es comprobar si, además del patrón histórico aprendido por el baseline, las variables disponibles antes del intervalo objetivo aportan señal adicional para estimar `ocupacion_pagada_proxy`.

La primera variable explicativa del modelo es `historical_profile_pred`, una predicción histórica derivada del baseline de perfiles. Esta variable se construye de forma distinta en entrenamiento y evaluación para evitar leakage. En las filas de evaluación de cada split, el perfil histórico se calcula ajustando el baseline exclusivamente con el tramo de entrenamiento correspondiente. En las filas de entrenamiento del modelo mínimo, no se utiliza una predicción in-sample calculada con todo el propio train; en su lugar, el tramo de entrenamiento se subdivide internamente en bloques cronológicos y la feature histórica de cada bloque se calcula usando únicamente los bloques anteriores. Las filas del primer bloque, al no disponer de histórico previo dentro del train, se excluyen del ajuste del modelo mínimo.

Junto a esta feature histórica, el modelo parte de una lista candidata de variables ex ante derivadas del panel. El código conserva únicamente las columnas presentes en `model_base` y excluye de forma explícita el target y las métricas SER contemporáneas del intervalo objetivo. La lista efectiva de predictores usados queda documentada en la tabla `minimum_model_feature_summary`, desglosada sin truncamiento para garantizar trazabilidad.

En esta formulación, las variables ex ante candidatas incluyen identificadores espaciales de barrio, variables temporales directas, codificaciones cíclicas de hora, día de semana y mes, flags temporales simples y variables estructurales de capacidad cuando están presentes en el panel final. No se incorporan variables cuya disponibilidad o join no haya quedado validado en el flujo previo.

No se incorporan como predictores el target ni las métricas SER contemporáneas del intervalo objetivo, ya que estas derivan de los mismos tiques utilizados para construir `ocupacion_pagada_proxy` y no estarían disponibles en una predicción futura real.

El estimador seleccionado para este modelo mínimo es Ridge, integrado en un pipeline con preprocesamiento diferenciado para variables numéricas y categóricas. Las variables numéricas se imputan y escalan; las variables categóricas se imputan y codifican mediante one-hot encoding con tratamiento de categorías no vistas. Ridge se utiliza por su simplicidad, reproducibilidad, bajo coste computacional y robustez frente a colinealidad, manteniendo una complejidad adecuada para evaluar si las variables ex ante aportan señal adicional sobre el perfil histórico.

La comparación relevante no es si este modelo obtiene un error bajo en términos absolutos, sino si mejora de forma consistente al baseline histórico bajo los mismos splits temporales, el mismo target y las mismas métricas de evaluación.


### 6.1 Justificación de variantes M1

La primera ejecución completa del modelo mínimo amplio permite evaluar si las variables ex ante mejoran al baseline histórico. No obstante, varias variables temporales candidatas codifican información ya capturada parcialmente por el baseline, especialmente el patrón por barrio, día de semana e intervalo horario. Por ello, antes de cerrar la lectura del notebook, se introduce un análisis de sensibilidad de features.

El objetivo de este análisis no es buscar un modelo complejo ni optimizar hiperparámetros, sino comprobar si el empeoramiento del modelo mínimo frente al baseline se debe a falta de señal adicional o a una especificación demasiado redundante. Se comparan tres variantes Ridge, todas entrenadas con validación temporal y con la misma feature histórica calculada sin leakage:

* `M1a_ridge_profile_only`: utiliza únicamente `historical_profile_pred`. Evalúa si una recalibración lineal del perfil histórico mejora o empeora el baseline.
* `M1b_ridge_profile_season_structural`: añade al perfil histórico variables estacionales y estructurales no contemporáneas, evitando reintroducir hora, día de semana e intervalo, que ya forman parte del baseline.
* `M1c_ridge_full_exante`: mantiene el set amplio inicial de variables ex ante. Esta variante sirve como contraste frente a una especificación más redundante.

La comparación relevante sigue siendo el skill frente a `M0_historical_profile`. Si ninguna variante mejora de forma consistente al baseline, el resultado se interpretará como evidencia de que el patrón histórico no paramétrico es una referencia más robusta para este proxy bajo la configuración actual.

### 6.2 Entrenamiento de variantes Ridge

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_expanding_historical_feature(
    train_df: pd.DataFrame,
    target_col: str,
    n_inner_blocks: int = 5,
    split_id: str | None = None,
) -> tuple[pd.Series, pd.DataFrame]:
    ordered_train = train_df.sort_values("intervalo_inicio")
    history_feature = pd.Series(np.nan, index=train_df.index, dtype="float64", name="historical_profile_pred")
    unique_timestamps = pd.Index(ordered_train["intervalo_inicio"].dropna().sort_values().unique())
    n_blocks = min(max(int(n_inner_blocks), 1), len(unique_timestamps))
    block_records = []

    if n_blocks < 2:
        block_records.append({
            "split_id": split_id,
            "inner_block": 1,
            "n_history_rows": 0,
            "n_predicted_rows": 0,
            "history_start": pd.NaT,
            "history_end": pd.NaT,
            "pred_start": ordered_train["intervalo_inicio"].min() if not ordered_train.empty else pd.NaT,
            "pred_end": ordered_train["intervalo_inicio"].max() if not ordered_train.empty else pd.NaT,
        })
        return history_feature, pd.DataFrame(block_records)

    timestamp_blocks = np.array_split(unique_timestamps.to_numpy(), n_blocks)
    block_by_timestamp = {
        timestamp: block_id
        for block_id, timestamp_block in enumerate(timestamp_blocks, start=1)
        for timestamp in timestamp_block
    }
    ordered_train = ordered_train.assign(_inner_block=ordered_train["intervalo_inicio"].map(block_by_timestamp))

    for block_id in range(1, n_blocks + 1):
        history_rows = ordered_train.loc[ordered_train["_inner_block"] < block_id]
        pred_rows = ordered_train.loc[ordered_train["_inner_block"] == block_id]
        n_predicted_rows = 0
        if block_id > 1 and not history_rows.empty and not pred_rows.empty:
            inner_baseline = HistoricalProfileBaseline().fit(history_rows.drop(columns=["_inner_block"]), target_col)
            inner_predictions = inner_baseline.predict(pred_rows.drop(columns=["_inner_block"]))
            history_feature.loc[pred_rows.index] = inner_predictions["y_pred"]
            n_predicted_rows = len(pred_rows)

        block_records.append({
            "split_id": split_id,
            "inner_block": block_id,
            "n_history_rows": len(history_rows),
            "n_predicted_rows": n_predicted_rows,
            "history_start": history_rows["intervalo_inicio"].min() if not history_rows.empty else pd.NaT,
            "history_end": history_rows["intervalo_inicio"].max() if not history_rows.empty else pd.NaT,
            "pred_start": pred_rows["intervalo_inicio"].min() if not pred_rows.empty else pd.NaT,
            "pred_end": pred_rows["intervalo_inicio"].max() if not pred_rows.empty else pd.NaT,
        })

    return history_feature, pd.DataFrame(block_records)


def build_minimum_model_pipeline(numeric_features: list[str], categorical_features: list[str]) -> Pipeline:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    transformers = []
    if numeric_features:
        transformers.append(("numeric", numeric_pipeline, numeric_features))
    if categorical_features:
        transformers.append(("categorical", categorical_pipeline, categorical_features))
    preprocessor = ColumnTransformer(transformers=transformers)
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0)),
    ])


def ordered_unique(values: list[str]) -> list[str]:
    return list(dict.fromkeys(values))


def sanitize_feature_list(candidate_features: list[str]) -> list[str]:
    return ordered_unique([
        feature for feature in candidate_features
        if feature == "historical_profile_pred"
        or (
            feature in model_base.columns
            and feature not in available_forbidden_feature_columns
            and feature != TARGET_COLUMN
            and feature not in LEAKAGE_COLUMNS
        )
    ])


def split_feature_types(feature_columns: list[str]) -> tuple[list[str], list[str]]:
    categorical_features = [
        col for col in ["barrio_key", "cod_distrito", "cod_barrio"]
        if col in feature_columns
    ]
    numeric_features = [col for col in feature_columns if col not in categorical_features]
    return numeric_features, categorical_features


def minimum_model_feature_group(feature: str) -> str:
    if feature == "historical_profile_pred":
        return "historical_profile"
    if feature in {"barrio_key", "cod_distrito", "cod_barrio"}:
        return "spatial_id"
    if feature in {"anio", "mes", "dia", "dia_semana_num", "hora", "minuto", "minuto_dia", "intervalo_30min_id"}:
        return "temporal_direct"
    if feature in {"sin_hora", "cos_hora", "sin_dia_semana", "cos_dia_semana", "sin_mes", "cos_mes"}:
        return "temporal_cyclical"
    if feature in {"es_agosto", "es_24_31_dic"}:
        return "temporal_flag"
    if feature in {"plazas_barrio_anio", "n_calles_barrio_anio"}:
        return "structural"
    return "other"


minimum_model_candidate_feature_sets = {
    "M1a_ridge_profile_only": ["historical_profile_pred"],
    "M1b_ridge_profile_season_structural": [
        "historical_profile_pred",
        "anio",
        "sin_mes",
        "cos_mes",
        "es_agosto",
        "es_24_31_dic",
        "plazas_barrio_anio",
    ],
    "M1c_ridge_full_exante": ["historical_profile_pred", *ex_ante_feature_candidates],
}
minimum_model_feature_sets = {
    model_id: sanitize_feature_list(candidate_features)
    for model_id, candidate_features in minimum_model_candidate_feature_sets.items()
}
minimum_model_feature_columns = ordered_unique([
    feature
    for feature_columns in minimum_model_feature_sets.values()
    for feature in feature_columns
])
minimum_model_numeric_features, minimum_model_categorical_features = split_feature_types(minimum_model_feature_columns)
minimum_model_feature_types = {
    model_id: split_feature_types(feature_columns)
    for model_id, feature_columns in minimum_model_feature_sets.items()
}

minimum_model_prediction_frames = []
minimum_model_check_records = []
minimum_model_summary_records = []
minimum_model_history_block_frames = []

for split_id, split_data in split_indices.items():
    definition = split_data["definition"]
    train_df = model_base.loc[split_data["train_idx"]].copy()
    eval_df = model_base.loc[split_data["eval_idx"]].copy()

    train_history_feature, history_blocks = build_expanding_historical_feature(
        train_df=train_df,
        target_col=TARGET_COLUMN,
        n_inner_blocks=5,
        split_id=split_id,
    )
    minimum_model_history_block_frames.append(history_blocks)

    eval_baseline = HistoricalProfileBaseline().fit(train_df, TARGET_COLUMN)
    eval_history_predictions = eval_baseline.predict(eval_df)

    train_model_df = train_df.copy()
    eval_model_df = eval_df.copy()
    train_model_df["historical_profile_pred"] = train_history_feature.reindex(train_model_df.index)
    eval_model_df["historical_profile_pred"] = eval_history_predictions["y_pred"].reindex(eval_model_df.index)

    train_usable_mask = train_model_df["historical_profile_pred"].notna()
    train_usable_df = train_model_df.loc[train_usable_mask].copy()
    n_train_rows_dropped_no_history = int((~train_usable_mask).sum())
    train_target = pd.to_numeric(train_usable_df[TARGET_COLUMN], errors="coerce")
    eval_target = pd.to_numeric(eval_model_df[TARGET_COLUMN], errors="coerce")

    for model_id, feature_columns in minimum_model_feature_sets.items():
        numeric_features, categorical_features = minimum_model_feature_types[model_id]
        forbidden_model_feature_overlap = sorted(set(feature_columns) & set(available_forbidden_feature_columns))
        leakage_model_feature_overlap = sorted(set(feature_columns) & set(LEAKAGE_COLUMNS))
        missing_model_features = [feature for feature in feature_columns if feature != "historical_profile_pred" and feature not in model_base.columns]

        check_values = {
            "feature_set_not_empty": (len(feature_columns) > 0, f"n_features={len(feature_columns)}"),
            "train_usable_not_empty": (not train_usable_df.empty, f"n_train_rows_used={len(train_usable_df)}"),
            "eval_not_empty": (not eval_model_df.empty, f"n_eval_rows={len(eval_model_df)}"),
            "train_usable_target_not_null": (train_target.isna().sum() == 0, f"missing_train_usable_target={train_target.isna().sum()}"),
            "eval_target_not_null": (eval_target.isna().sum() == 0, f"missing_eval_target={eval_target.isna().sum()}"),
            "train_history_feature_not_null": (train_usable_df["historical_profile_pred"].isna().sum() == 0, f"missing_train_history={train_usable_df['historical_profile_pred'].isna().sum()}"),
            "eval_history_feature_not_null": (eval_model_df["historical_profile_pred"].isna().sum() == 0, f"missing_eval_history={eval_model_df['historical_profile_pred'].isna().sum()}"),
            "no_forbidden_features": (not forbidden_model_feature_overlap, f"forbidden_feature_overlap={forbidden_model_feature_overlap}"),
            "no_leakage_features": (not leakage_model_feature_overlap, f"leakage_feature_overlap={leakage_model_feature_overlap}"),
            "target_not_in_features": (TARGET_COLUMN not in feature_columns, f"target_in_features={TARGET_COLUMN in feature_columns}"),
            "all_features_available": (not missing_model_features, f"missing_model_features={missing_model_features}"),
        }

        can_fit_predict = all(value for value, _detail in check_values.values())
        if can_fit_predict:
            minimum_model_pipeline = build_minimum_model_pipeline(numeric_features, categorical_features)
            minimum_model_pipeline.fit(train_usable_df[feature_columns], train_target)
            y_pred = pd.Series(
                minimum_model_pipeline.predict(eval_model_df[feature_columns]),
                index=eval_model_df.index,
                dtype="float64",
            )
        else:
            y_pred = pd.Series(np.nan, index=eval_model_df.index, dtype="float64")

        prediction_errors = y_pred - eval_target
        split_predictions = pd.DataFrame({
            "model_id": model_id,
            "split_id": split_id,
            "eval_label": definition["eval_label"],
            "barrio_key": eval_model_df["barrio_key"],
            "intervalo_inicio": eval_model_df["intervalo_inicio"],
            "y_true": eval_target,
            "y_pred": y_pred,
            "error": prediction_errors,
            "abs_error": prediction_errors.abs(),
            "historical_profile_pred": eval_model_df["historical_profile_pred"],
        })
        minimum_model_prediction_frames.append(split_predictions)

        pred_missing = y_pred.isna().sum()
        pred_non_finite = (~np.isfinite(y_pred.to_numpy(dtype=float))).sum() - pred_missing
        check_values.update({
            "n_predictions_matches_eval": (len(split_predictions) == len(eval_model_df), f"n_predictions={len(split_predictions)}; n_eval_rows={len(eval_model_df)}"),
            "predictions_not_null": (pred_missing == 0, f"missing_predictions={pred_missing}"),
            "predictions_finite": (pred_non_finite == 0, f"non_finite_predictions={pred_non_finite}"),
        })
        for check_name, (check_ok, detail) in check_values.items():
            minimum_model_check_records.append({
                "model_id": model_id,
                "split_id": split_id,
                "check": check_name,
                "status": "OK" if check_ok else "FAIL",
                "detail": detail,
            })

        minimum_model_summary_records.append({
            "model_id": model_id,
            "split_id": split_id,
            "eval_label": definition["eval_label"],
            "n_train_rows_total": len(train_model_df),
            "n_train_rows_used": len(train_usable_df),
            "n_train_rows_dropped_no_history": n_train_rows_dropped_no_history,
            "n_eval_rows": len(eval_model_df),
            "n_predictions": len(split_predictions),
            "n_features": len(feature_columns),
            "n_numeric_features": len(numeric_features),
            "n_categorical_features": len(categorical_features),
            "eval_target_mean": eval_target.mean(),
            "pred_mean": y_pred.mean(),
            "mae": split_predictions["abs_error"].mean(),
            "rmse": np.sqrt(np.mean(np.square(split_predictions["error"]))),
            "bias": prediction_errors.mean(),
        })

minimum_model_predictions = pd.concat(minimum_model_prediction_frames, ignore_index=True)
minimum_model_checks = pd.DataFrame(minimum_model_check_records)
minimum_model_history_feature_blocks = pd.concat(minimum_model_history_block_frames, ignore_index=True)
minimum_model_status = (
    minimum_model_checks.assign(status_rank=lambda df: df["status"].map({"OK": 0, "WARNING": 1, "FAIL": 2}))
    .groupby(["model_id", "split_id"], as_index=False)["status_rank"]
    .max()
    .assign(checks_status=lambda df: df["status_rank"].map({0: "OK", 1: "WARNING", 2: "FAIL"}))
    [["model_id", "split_id", "checks_status"]]
)
minimum_model_summary = pd.DataFrame(minimum_model_summary_records).merge(minimum_model_status, on=["model_id", "split_id"], how="left")
minimum_model_feature_overview = pd.DataFrame([
    {
        "model_id": model_id,
        "n_features": len(feature_columns),
        "n_numeric_features": len(minimum_model_feature_types[model_id][0]),
        "n_categorical_features": len(minimum_model_feature_types[model_id][1]),
    }
    for model_id, feature_columns in minimum_model_feature_sets.items()
])
minimum_model_feature_summary = pd.DataFrame([
    {
        "model_id": model_id,
        "feature_order": feature_order,
        "feature": feature,
        "feature_type": "categorical" if feature in minimum_model_feature_types[model_id][1] else "numeric",
        "feature_group": minimum_model_feature_group(feature),
        "used_in_model": True,
    }
    for model_id, feature_columns in minimum_model_feature_sets.items()
    for feature_order, feature in enumerate(feature_columns, start=1)
])

display(minimum_model_summary)
display(minimum_model_feature_overview)
display(minimum_model_feature_summary)

minimum_model_checks_with_issues = minimum_model_checks.loc[minimum_model_checks["status"].isin(["WARNING", "FAIL"])]
if SHOW_DEBUG_TABLES or not minimum_model_checks_with_issues.empty:
    display(minimum_model_checks)
if SHOW_DEBUG_TABLES or not minimum_model_checks_with_issues.empty:
    display(minimum_model_history_feature_blocks)

failed_minimum_model_checks = minimum_model_checks.loc[minimum_model_checks["status"].eq("FAIL")]
if not failed_minimum_model_checks.empty:
    raise ValueError(f"Checks de modelo mínimo fallidos: {failed_minimum_model_checks[['model_id', 'split_id', 'check']].to_dict('records')}")


,model_id,split_id,eval_label,n_train_rows_total,n_train_rows_used,n_train_rows_dropped_no_history,n_eval_rows,n_predictions,n_features,n_numeric_features,n_categorical_features,eval_target_mean,pred_mean,mae,rmse,bias,checks_status
0,M1a_ridge_profile_only,split_validation,validation,775008,623808,151200,410280,410280,1,1,0,0.097507,0.091808,0.017085,0.025680,-0.005700,OK
1,M1b_ridge_profile_season_structural,split_validation,validation,775008,623808,151200,410280,410280,7,7,0,0.097507,0.087659,0.019184,0.027340,-0.009849,OK
2,M1c_ridge_full_exante,split_validation,validation,775008,623808,151200,410280,410280,19,18,1,0.097507,0.089946,0.018896,0.029497,-0.007562,OK
3,M1a_ridge_profile_only,split_test_refit,test,1185288,958308,226980,106860,106860,1,1,0,0.098564,0.095480,0.011742,0.015559,-0.003084,OK
4,M1b_ridge_profile_season_structural,split_test_refit,test,1185288,958308,226980,106860,106860,7,7,0,0.098564,0.106256,0.014010,0.017216,0.007692,OK
5,M1c_ridge_full_exante,split_test_refit,test,1185288,958308,226980,106860,106860,19,18,1,0.098564,0.104137,0.013777,0.017999,0.005572,OK


,model_id,n_features,n_numeric_features,n_categorical_features
0,M1a_ridge_profile_only,1,1,0
1,M1b_ridge_profile_season_structural,7,7,0
2,M1c_ridge_full_exante,19,18,1


,model_id,feature_order,feature,feature_type,feature_group,used_in_model
0,M1a_ridge_profile_only,1,historical_profile_pred,numeric,historical_profile,True
1,M1b_ridge_profile_season_structural,1,historical_profile_pred,numeric,historical_profile,True
2,M1b_ridge_profile_season_structural,2,anio,numeric,temporal_direct,True
3,M1b_ridge_profile_season_structural,3,sin_mes,numeric,temporal_cyclical,True
4,M1b_ridge_profile_season_structural,4,cos_mes,numeric,temporal_cyclical,True
5,M1b_ridge_profile_season_structural,5,es_agosto,numeric,temporal_flag,True
6,M1b_ridge_profile_season_structural,6,es_24_31_dic,numeric,temporal_flag,True
7,M1b_ridge_profile_season_structural,7,plazas_barrio_anio,numeric,structural,True
8,M1c_ridge_full_exante,1,historical_profile_pred,numeric,historical_profile,True
9,M1c_ridge_full_exante,2,barrio_key,categorical,spatial_id,True


Las variantes Ridge se entrenan correctamente en ambos splits y todos los checks finalizan en estado `OK`. Esto confirma que el modelo mínimo se ha construido respetando el contrato metodológico: la feature histórica se calcula sin predicción in-sample directa, el target y las métricas contemporáneas quedan excluidos de los predictores, y las predicciones sobre validación y test son completas y finitas.

La construcción de `historical_profile_pred` implica descartar del ajuste del modelo mínimo las filas iniciales del entrenamiento que no disponen de histórico previo dentro del esquema expansivo. En `split_validation` se utilizan 623.808 de las 775.008 filas de entrenamiento, y en `split_test_refit` se utilizan 958.308 de 1.185.288. Esta pérdida de filas no afecta a la evaluación, ya que las predicciones se generan sobre todas las observaciones de validación y test, pero debe tenerse en cuenta al interpretar las variantes M1.

La variante `M1a_ridge_profile_only`, basada únicamente en `historical_profile_pred`, es la especificación más limpia del modelo mínimo. Su función es comprobar si una recalibración lineal del perfil histórico mejora al baseline no paramétrico. En validación reduce el MAE respecto a M0 de 0,0177 a 0,0171, y en test lo reduce de 0,0132 a 0,0117. También reduce el RMSE y el sesgo medio en ambos splits. Por tanto, desde una lectura global, M1a aporta una mejora moderada frente al baseline histórico.

Las variantes con más variables no muestran el mismo comportamiento. `M1b_ridge_profile_season_structural`, que añade estacionalidad anual y capacidad estructural, empeora el MAE en validación y en test. `M1c_ridge_full_exante`, que incorpora el conjunto amplio de variables ex ante, tampoco mejora de forma robusta al baseline. Estos resultados sugieren que añadir variables temporales y estructurales bajo una especificación lineal no garantiza una mejora predictiva sobre el perfil histórico.

La lectura metodológica es que el patrón histórico por barrio, día de semana e intervalo horario concentra gran parte de la señal útil para este proxy. Las variables adicionales pueden ser razonables desde el punto de vista conceptual, pero en esta configuración no aportan una ganancia suficientemente consistente. Por ello, M1b y M1c deben interpretarse como análisis de sensibilidad, no como especificaciones finales optimizadas.

La decisión definitiva no debe tomarse únicamente con las métricas agregadas de esta sección, sino con la evaluación común de la Sección 7, donde se comparan todos los modelos frente a M0 bajo las mismas métricas, incluyendo el comportamiento en valores altos del target.


## 7. Evaluación común y métricas

La evaluación compara el baseline histórico (`M0_historical_profile`) y las variantes del modelo mínimo (`M1a_ridge_profile_only`, `M1b_ridge_profile_season_structural` y `M1c_ridge_full_exante`) bajo los mismos splits temporales, el mismo target y las mismas observaciones de evaluación. De este modo, las diferencias de rendimiento se atribuyen al modelo y a las variables utilizadas, no a cambios en la muestra evaluada.

La primera tabla de salida, `metrics_comparison`, recoge las métricas principales por modelo y split. Las métricas generales son MAE, RMSE, R² y bias medio. No se utiliza MAPE porque el target puede tomar valores cercanos a cero, lo que haría la métrica inestable y difícil de interpretar.

Además de las métricas globales, se evalúa el comportamiento en valores altos del target. Para ello, en cada split se calcula un umbral de target alto a partir del percentil 90 de `ocupacion_pagada_proxy` en el tramo de entrenamiento. Este umbral se calcula exclusivamente con train, sin utilizar validación ni test. Sobre las filas de evaluación cuyo target supera ese umbral se calculan métricas específicas de error, como `mae_high_target` y `rmse_high_target`.

La comparación frente al baseline se resume mediante skill scores. Para cada split, el baseline histórico actúa como referencia y el modelo mínimo se evalúa mediante:

```text
mae_skill_vs_baseline = 1 - MAE(M1) / MAE(M0)
rmse_skill_vs_baseline = 1 - RMSE(M1) / RMSE(M0)
```

Un skill positivo indica que el modelo mínimo mejora al baseline; un skill negativo indica que empeora frente a la referencia histórica. Para el propio baseline, el skill se fija en 0 como punto de comparación.

La segunda tabla de salida, `model_comparison_delta`, resume directamente la diferencia entre el modelo candidato y el modelo de referencia dentro de cada split. La comparación se define como:

```text
candidate_model ∈ {M1a_ridge_profile_only, M1b_ridge_profile_season_structural, M1c_ridge_full_exante}
reference_model = M0_historical_profile
delta_definition = candidate_minus_reference
```

Por tanto, los deltas se interpretan como:

```text
delta_mae = MAE(M1) - MAE(M0)
delta_rmse = RMSE(M1) - RMSE(M0)
delta_r2 = R²(M1) - R²(M0)
delta_abs_bias = |bias(M1)| - |bias(M0)|
```

En consecuencia, si `delta_mae < 0`, el modelo mínimo mejora al baseline en MAE; si `delta_mae > 0`, empeora frente al baseline. De forma equivalente, un `mae_skill_vs_baseline > 0` indica mejora de M1 frente a M0, mientras que un valor negativo indica peor rendimiento que el baseline.

El código también calcula errores agregados por barrio y por hora para detectar patrones espaciales o temporales que puedan quedar ocultos en las métricas medias. Estas tablas no se muestran por defecto para evitar saturar el notebook, pero quedan disponibles internamente y pueden revisarse activando `SHOW_DEBUG_TABLES`.

Los resultados obtenidos en modo `smoke` no deben interpretarse como rendimiento final. Solo sirven para comprobar que la evaluación, los cálculos de métricas, los umbrales, los skill scores y las tablas comparativas se generan correctamente antes de ejecutar el notebook completo.


In [18]:
required_prediction_columns = [
    "model_id",
    "split_id",
    "eval_label",
    "barrio_key",
    "intervalo_inicio",
    "y_true",
    "y_pred",
    "error",
    "abs_error",
]

candidate_model_ids = [
    "M1a_ridge_profile_only",
    "M1b_ridge_profile_season_structural",
    "M1c_ridge_full_exante",
]
reference_model_id = "M0_historical_profile"

evaluation_predictions = pd.concat([baseline_predictions, minimum_model_predictions], ignore_index=True)
missing_prediction_columns = [col for col in required_prediction_columns if col not in evaluation_predictions.columns]
if missing_prediction_columns:
    raise ValueError(f"Faltan columnas requeridas en predicciones: {missing_prediction_columns}")

target_high_quantile = 0.90
high_target_threshold_records = []
for split_id, split_data in split_indices.items():
    train_target = pd.to_numeric(model_base.loc[split_data["train_idx"], TARGET_COLUMN], errors="coerce")
    train_target_finite = train_target[np.isfinite(train_target.to_numpy(dtype=float))]
    high_target_threshold_records.append({
        "split_id": split_id,
        "target_high_quantile": target_high_quantile,
        "target_high_threshold": train_target_finite.quantile(target_high_quantile) if not train_target_finite.empty else np.nan,
        "n_train_rows_for_threshold": len(train_target_finite),
    })
high_target_thresholds = pd.DataFrame(high_target_threshold_records)


def compute_regression_metrics(df: pd.DataFrame) -> dict:
    y_true = pd.to_numeric(df["y_true"], errors="coerce").to_numpy(dtype=float)
    y_pred = pd.to_numeric(df["y_pred"], errors="coerce").to_numpy(dtype=float)
    error = y_pred - y_true
    sse = np.sum(np.square(error))
    sst = np.sum(np.square(y_true - np.mean(y_true)))
    return {
        "n_obs": len(df),
        "y_true_mean": np.mean(y_true),
        "y_pred_mean": np.mean(y_pred),
        "mae": np.mean(np.abs(error)),
        "rmse": np.sqrt(np.mean(np.square(error))),
        "r2": np.nan if sst == 0 else 1 - sse / sst,
        "bias": np.mean(error),
    }


evaluation_metric_records = []
for (split_id, eval_label, model_id), group in evaluation_predictions.groupby(["split_id", "eval_label", "model_id"], dropna=False):
    metric_record = {"split_id": split_id, "eval_label": eval_label, "model_id": model_id}
    metric_record.update(compute_regression_metrics(group))
    evaluation_metric_records.append(metric_record)
evaluation_metrics = pd.DataFrame(evaluation_metric_records)

high_target_metric_records = []
for (split_id, eval_label, model_id), group in evaluation_predictions.groupby(["split_id", "eval_label", "model_id"], dropna=False):
    threshold_row = high_target_thresholds.loc[high_target_thresholds["split_id"].eq(split_id)]
    threshold = threshold_row["target_high_threshold"].iloc[0] if not threshold_row.empty else np.nan
    high_group = group.loc[pd.to_numeric(group["y_true"], errors="coerce") >= threshold]
    if high_group.empty:
        high_target_metric_records.append({
            "split_id": split_id,
            "eval_label": eval_label,
            "model_id": model_id,
            "n_high_target": 0,
            "mae_high_target": np.nan,
            "rmse_high_target": np.nan,
            "bias_high_target": np.nan,
        })
    else:
        high_metrics = compute_regression_metrics(high_group)
        high_target_metric_records.append({
            "split_id": split_id,
            "eval_label": eval_label,
            "model_id": model_id,
            "n_high_target": high_metrics["n_obs"],
            "mae_high_target": high_metrics["mae"],
            "rmse_high_target": high_metrics["rmse"],
            "bias_high_target": high_metrics["bias"],
        })
high_target_metrics = pd.DataFrame(high_target_metric_records)
evaluation_metrics = evaluation_metrics.merge(high_target_metrics, on=["split_id", "eval_label", "model_id"], how="left")

baseline_error_reference = evaluation_metrics.loc[
    evaluation_metrics["model_id"].eq(reference_model_id),
    ["split_id", "mae", "rmse"],
].rename(columns={"mae": "mae_baseline", "rmse": "rmse_baseline"})
evaluation_metrics = evaluation_metrics.merge(baseline_error_reference, on="split_id", how="left")
evaluation_metrics["mae_skill_vs_baseline"] = np.where(
    evaluation_metrics["model_id"].eq(reference_model_id),
    0.0,
    np.where(evaluation_metrics["mae_baseline"].gt(0), 1 - evaluation_metrics["mae"] / evaluation_metrics["mae_baseline"], np.nan),
)
evaluation_metrics["rmse_skill_vs_baseline"] = np.where(
    evaluation_metrics["model_id"].eq(reference_model_id),
    0.0,
    np.where(evaluation_metrics["rmse_baseline"].gt(0), 1 - evaluation_metrics["rmse"] / evaluation_metrics["rmse_baseline"], np.nan),
)
baseline_high_target_reference = evaluation_metrics.loc[
    evaluation_metrics["model_id"].eq(reference_model_id),
    ["split_id", "mae_high_target", "rmse_high_target"],
].rename(columns={"mae_high_target": "mae_high_target_baseline", "rmse_high_target": "rmse_high_target_baseline"})
evaluation_metrics = evaluation_metrics.merge(baseline_high_target_reference, on="split_id", how="left")
evaluation_metrics["mae_high_target_skill_vs_baseline"] = np.where(
    evaluation_metrics["model_id"].eq(reference_model_id),
    0.0,
    np.where(
        np.isfinite(evaluation_metrics["mae_high_target_baseline"]) & evaluation_metrics["mae_high_target_baseline"].gt(0),
        1 - evaluation_metrics["mae_high_target"] / evaluation_metrics["mae_high_target_baseline"],
        np.nan,
    ),
)
evaluation_metrics["rmse_high_target_skill_vs_baseline"] = np.where(
    evaluation_metrics["model_id"].eq(reference_model_id),
    0.0,
    np.where(
        np.isfinite(evaluation_metrics["rmse_high_target_baseline"]) & evaluation_metrics["rmse_high_target_baseline"].gt(0),
        1 - evaluation_metrics["rmse_high_target"] / evaluation_metrics["rmse_high_target_baseline"],
        np.nan,
    ),
)
evaluation_metrics = evaluation_metrics.drop(columns=["mae_baseline", "rmse_baseline", "mae_high_target_baseline", "rmse_high_target_baseline"])

metrics_comparison = (
    evaluation_metrics.merge(high_target_thresholds[["split_id", "target_high_threshold"]], on="split_id", how="left")
    [[
        "split_id",
        "eval_label",
        "model_id",
        "n_obs",
        "n_high_target",
        "target_high_threshold",
        "mae",
        "rmse",
        "r2",
        "bias",
        "mae_high_target",
        "rmse_high_target",
        "mae_skill_vs_baseline",
        "rmse_skill_vs_baseline",
        "mae_high_target_skill_vs_baseline",
        "rmse_high_target_skill_vs_baseline",
    ]]
    .sort_values(["split_id", "model_id"])
    .reset_index(drop=True)
)

model_comparison_delta_records = []
for split_id, split_group in evaluation_metrics.groupby("split_id", dropna=False):
    baseline_row = split_group.loc[split_group["model_id"].eq(reference_model_id)]
    if baseline_row.empty:
        continue
    baseline_row = baseline_row.iloc[0]
    for candidate_model in sorted(model for model in split_group["model_id"].unique() if model != reference_model_id):
        model_row = split_group.loc[split_group["model_id"].eq(candidate_model)].iloc[0]
        delta_mae = model_row["mae"] - baseline_row["mae"]
        delta_mae_high_target = model_row["mae_high_target"] - baseline_row["mae_high_target"]
        model_comparison_delta_records.append({
            "split_id": split_id,
            "eval_label": model_row["eval_label"],
            "candidate_model": candidate_model,
            "reference_model": reference_model_id,
            "delta_definition": "candidate_minus_reference",
            "delta_mae": delta_mae,
            "delta_rmse": model_row["rmse"] - baseline_row["rmse"],
            "delta_r2": model_row["r2"] - baseline_row["r2"],
            "delta_abs_bias": abs(model_row["bias"]) - abs(baseline_row["bias"]),
            "delta_mae_high_target": delta_mae_high_target,
            "delta_rmse_high_target": model_row["rmse_high_target"] - baseline_row["rmse_high_target"],
            "delta_abs_bias_high_target": abs(model_row["bias_high_target"]) - abs(baseline_row["bias_high_target"]),
            "mae_skill_vs_baseline": model_row["mae_skill_vs_baseline"],
            "rmse_skill_vs_baseline": model_row["rmse_skill_vs_baseline"],
            "mae_high_target_skill_vs_baseline": model_row["mae_high_target_skill_vs_baseline"],
            "rmse_high_target_skill_vs_baseline": model_row["rmse_high_target_skill_vs_baseline"],
            "preferred_model_by_mae": candidate_model if delta_mae < 0 else reference_model_id,
            "preferred_model_by_high_target_mae": candidate_model if delta_mae_high_target < 0 else reference_model_id,
        })
model_comparison_delta = pd.DataFrame(model_comparison_delta_records)[[
    "split_id",
    "eval_label",
    "candidate_model",
    "reference_model",
    "delta_definition",
    "delta_mae",
    "delta_rmse",
    "delta_r2",
    "delta_abs_bias",
    "delta_mae_high_target",
    "delta_rmse_high_target",
    "delta_abs_bias_high_target",
    "mae_skill_vs_baseline",
    "rmse_skill_vs_baseline",
    "mae_high_target_skill_vs_baseline",
    "rmse_high_target_skill_vs_baseline",
    "preferred_model_by_mae",
    "preferred_model_by_high_target_mae",
]]

if "hora" not in evaluation_predictions.columns:
    evaluation_predictions["hora"] = pd.to_datetime(evaluation_predictions["intervalo_inicio"], errors="coerce").dt.hour


def grouped_error_metrics(group: pd.DataFrame) -> pd.Series:
    metrics = compute_regression_metrics(group)
    return pd.Series({
        "n_obs": metrics["n_obs"],
        "mae": metrics["mae"],
        "rmse": metrics["rmse"],
        "bias": metrics["bias"],
    })


error_by_barrio = (
    evaluation_predictions.groupby(["split_id", "model_id", "barrio_key"], dropna=False)
    .apply(grouped_error_metrics)
    .reset_index()
)
error_by_hour = (
    evaluation_predictions.groupby(["split_id", "model_id", "hora"], dropna=False)
    .apply(grouped_error_metrics)
    .reset_index()
)

evaluation_check_records = []
baseline_prediction_exists = "baseline_predictions" in globals() and not baseline_predictions.empty
minimum_model_prediction_exists = "minimum_model_predictions" in globals() and not minimum_model_predictions.empty
evaluation_check_records.append({"check": "baseline_predictions_exist", "status": "OK" if baseline_prediction_exists else "FAIL", "detail": f"n_rows={len(baseline_predictions) if 'baseline_predictions' in globals() else 0}"})
evaluation_check_records.append({"check": "minimum_model_predictions_exist", "status": "OK" if minimum_model_prediction_exists else "FAIL", "detail": f"n_rows={len(minimum_model_predictions) if 'minimum_model_predictions' in globals() else 0}"})
evaluation_check_records.append({"check": "required_prediction_columns_present", "status": "OK" if not missing_prediction_columns else "FAIL", "detail": f"missing={missing_prediction_columns}"})
y_true_values = pd.to_numeric(evaluation_predictions["y_true"], errors="coerce")
y_pred_values = pd.to_numeric(evaluation_predictions["y_pred"], errors="coerce")
evaluation_check_records.append({"check": "y_true_not_null", "status": "OK" if y_true_values.isna().sum() == 0 else "FAIL", "detail": f"missing_y_true={y_true_values.isna().sum()}"})
evaluation_check_records.append({"check": "y_pred_not_null", "status": "OK" if y_pred_values.isna().sum() == 0 else "FAIL", "detail": f"missing_y_pred={y_pred_values.isna().sum()}"})
evaluation_check_records.append({"check": "y_pred_finite", "status": "OK" if np.isfinite(y_pred_values.to_numpy(dtype=float)).all() else "FAIL", "detail": f"non_finite_y_pred={(~np.isfinite(y_pred_values.to_numpy(dtype=float))).sum()}"})
evaluation_check_records.append({"check": "y_true_finite", "status": "OK" if np.isfinite(y_true_values.to_numpy(dtype=float)).all() else "FAIL", "detail": f"non_finite_y_true={(~np.isfinite(y_true_values.to_numpy(dtype=float))).sum()}"})

expected_split_ids = [definition["split_id"] for definition in split_definitions]
for split_id in expected_split_ids:
    split_models = set(evaluation_predictions.loc[evaluation_predictions["split_id"].eq(split_id), "model_id"])
    evaluation_check_records.append({"check": f"{split_id}_has_baseline", "status": "OK" if reference_model_id in split_models else "FAIL", "detail": f"models={sorted(split_models)}"})
    missing_candidate_models = [model_id for model_id in candidate_model_ids if model_id not in split_models]
    evaluation_check_records.append({"check": f"{split_id}_has_all_candidate_models", "status": "OK" if not missing_candidate_models else "FAIL", "detail": f"missing={missing_candidate_models}; models={sorted(split_models)}"})

threshold_failures = high_target_thresholds.loc[
    high_target_thresholds["n_train_rows_for_threshold"].eq(0) | ~np.isfinite(high_target_thresholds["target_high_threshold"].to_numpy(dtype=float))
]
evaluation_check_records.append({"check": "target_high_threshold_from_train", "status": "OK" if threshold_failures.empty else "FAIL", "detail": f"failed_splits={threshold_failures['split_id'].tolist()}"})

high_target_by_split = high_target_metrics.groupby("split_id", as_index=False)["n_high_target"].max()
no_high_target_splits = high_target_by_split.loc[high_target_by_split["n_high_target"].eq(0), "split_id"].tolist()
evaluation_check_records.append({"check": "n_high_target_positive_by_split", "status": "OK" if not no_high_target_splits else "WARNING", "detail": f"splits_without_high_target={no_high_target_splits}"})
duplicate_prediction_count = evaluation_predictions.duplicated(["model_id", "split_id", "barrio_key", "intervalo_inicio"]).sum()
evaluation_check_records.append({"check": "no_duplicate_model_split_barrio_interval", "status": "OK" if duplicate_prediction_count == 0 else "FAIL", "detail": f"duplicate_rows={duplicate_prediction_count}"})

baseline_zero_error_splits = baseline_error_reference.loc[
    baseline_error_reference["mae_baseline"].isna()
    | baseline_error_reference["rmse_baseline"].isna()
    | baseline_error_reference["mae_baseline"].le(0)
    | baseline_error_reference["rmse_baseline"].le(0),
    "split_id",
].tolist()
evaluation_check_records.append({"check": "baseline_error_positive_for_skill", "status": "OK" if not baseline_zero_error_splits else "WARNING", "detail": f"splits_with_zero_baseline_error={baseline_zero_error_splits}"})

evaluation_checks = pd.DataFrame(evaluation_check_records)

display(metrics_comparison)
display(model_comparison_delta)

evaluation_checks_with_issues = evaluation_checks.loc[evaluation_checks["status"].isin(["WARNING", "FAIL"])]
if SHOW_DEBUG_TABLES or not evaluation_checks_with_issues.empty:
    display(evaluation_checks)
if SHOW_DEBUG_TABLES or not evaluation_checks_with_issues.empty:
    display(error_by_barrio)
    display(error_by_hour)

failed_evaluation_checks = evaluation_checks.loc[evaluation_checks["status"].eq("FAIL")]
if not failed_evaluation_checks.empty:
    raise ValueError(f"Checks de evaluación fallidos: {failed_evaluation_checks[['check', 'detail']].to_dict('records')}")


,split_id,eval_label,model_id,n_obs,n_high_target,target_high_threshold,mae,rmse,r2,bias,mae_high_target,rmse_high_target,mae_skill_vs_baseline,rmse_skill_vs_baseline,mae_high_target_skill_vs_baseline,rmse_high_target_skill_vs_baseline
0,split_test_refit,test,M0_historical_profile,106860,8008,0.158345,0.013168,0.017864,0.803861,-0.006815,0.015392,0.020393,0.000000,0.000000,0.000000,0.000000
1,split_test_refit,test,M1a_ridge_profile_only,106860,8008,0.158345,0.011742,0.015559,0.851201,-0.003084,0.020327,0.025423,0.108311,0.129001,-0.320636,-0.246694
2,split_test_refit,test,M1b_ridge_profile_season_structural,106860,8008,0.158345,0.014010,0.017216,0.817818,0.007692,0.014305,0.019004,-0.063921,0.036238,0.070612,0.068097
3,split_test_refit,test,M1c_ridge_full_exante,106860,8008,0.158345,0.013777,0.017999,0.800874,0.005572,0.012851,0.017379,-0.046257,-0.007586,0.165058,0.147789
4,split_validation,validation,M0_historical_profile,410280,39523,0.158695,0.017702,0.026926,0.639862,-0.006910,0.019112,0.025676,0.000000,0.000000,0.000000,0.000000
5,split_validation,validation,M1a_ridge_profile_only,410280,39523,0.158695,0.017085,0.025680,0.672441,-0.005700,0.025507,0.031127,0.034850,0.046303,-0.334634,-0.212314
6,split_validation,validation,M1b_ridge_profile_season_structural,410280,39523,0.158695,0.019184,0.027340,0.628712,-0.009849,0.029892,0.036130,-0.083767,-0.015363,-0.564044,-0.407180
7,split_validation,validation,M1c_ridge_full_exante,410280,39523,0.158695,0.018896,0.029497,0.567817,-0.007562,0.025606,0.032741,-0.067476,-0.095467,-0.339811,-0.275188


,split_id,eval_label,candidate_model,reference_model,delta_definition,delta_mae,delta_rmse,delta_r2,delta_abs_bias,delta_mae_high_target,delta_rmse_high_target,delta_abs_bias_high_target,mae_skill_vs_baseline,rmse_skill_vs_baseline,mae_high_target_skill_vs_baseline,rmse_high_target_skill_vs_baseline,preferred_model_by_mae,preferred_model_by_high_target_mae
0,split_test_refit,test,M1a_ridge_profile_only,M0_historical_profile,candidate_minus_reference,-0.001426,-0.002304,0.047340,-0.003731,0.004935,0.005031,0.009234,0.108311,0.129001,-0.320636,-0.246694,M1a_ridge_profile_only,M0_historical_profile
1,split_test_refit,test,M1b_ridge_profile_season_structural,M0_historical_profile,candidate_minus_reference,0.000842,-0.000647,0.013958,0.000877,-0.001087,-0.001389,-0.001646,-0.063921,0.036238,0.070612,0.068097,M0_historical_profile,M1b_ridge_profile_season_structural
2,split_test_refit,test,M1c_ridge_full_exante,M0_historical_profile,candidate_minus_reference,0.000609,0.000136,-0.002987,-0.001243,-0.002541,-0.003014,-0.004396,-0.046257,-0.007586,0.165058,0.147789,M0_historical_profile,M1c_ridge_full_exante
3,split_validation,validation,M1a_ridge_profile_only,M0_historical_profile,candidate_minus_reference,-0.000617,-0.001247,0.032579,-0.001210,0.006395,0.005451,0.008774,0.034850,0.046303,-0.334634,-0.212314,M1a_ridge_profile_only,M0_historical_profile
4,split_validation,validation,M1b_ridge_profile_season_structural,M0_historical_profile,candidate_minus_reference,0.001483,0.000414,-0.011150,0.002939,0.010780,0.010455,0.013233,-0.083767,-0.015363,-0.564044,-0.407180,M0_historical_profile,M0_historical_profile
5,split_validation,validation,M1c_ridge_full_exante,M0_historical_profile,candidate_minus_reference,0.001194,0.002571,-0.072045,0.000652,0.006494,0.007066,0.008336,-0.067476,-0.095467,-0.339811,-0.275188,M0_historical_profile,M0_historical_profile


La evaluación común confirma que el baseline histórico M0 es una referencia fuerte para el problema planteado. En validación, M0 obtiene un MAE de 0,0177, RMSE de 0,0269 y R² de 0,6399. En test, mejora hasta un MAE de 0,0132, RMSE de 0,0179 y R² de 0,8039. Estos valores indican que el patrón histórico por barrio, día de semana e intervalo horario captura una parte relevante de la variabilidad del proxy.

La variante `M1a_ridge_profile_only` mejora de forma consistente las métricas globales frente a M0. En validación reduce el MAE de 0,0177 a 0,0171, el RMSE de 0,0269 a 0,0257 y eleva el R² de 0,6399 a 0,6724. En test, reduce el MAE de 0,0132 a 0,0117, el RMSE de 0,0179 a 0,0156 y eleva el R² de 0,8039 a 0,8512. También reduce el sesgo absoluto en ambos splits. Desde una lectura puramente global, M1a es la variante con mejor rendimiento agregado.

Sin embargo, el análisis sobre valores altos del target introduce una cautela importante. En validación, M1a empeora el MAE en target alto respecto a M0, pasando de 0,0191 a 0,0255. En test ocurre lo mismo: el MAE en target alto pasa de 0,0154 a 0,0203. El skill negativo de M1a en este subconjunto indica que la recalibración lineal mejora el ajuste medio, pero degrada precisamente los episodios de mayor presión pagada relativa, que son los más relevantes para una lectura de dificultad de aparcamiento.

Las variantes `M1b_ridge_profile_season_structural` y `M1c_ridge_full_exante` tampoco ofrecen una mejora robusta. M1b empeora el MAE global en validación y test, aunque en test mejora ligeramente el comportamiento sobre target alto. M1c mejora el MAE de target alto en test, pero no reproduce esa mejora en validación, donde empeora frente a M0. Por tanto, no sería metodológicamente correcto seleccionar M1c por su resultado en test, ya que la validación temporal previa no respalda de forma consistente esa mejora.

La tabla de deltas refuerza esta lectura. M1a es preferible a M0 por MAE global en ambos splits, pero M0 es preferible por MAE en target alto tanto en validación como en test. M1b y M1c no muestran una mejora global estable, y sus ventajas parciales en target alto no son consistentes entre splits. En consecuencia, la evidencia no justifica sustituir el baseline histórico por una variante lineal más compleja como modelo principal de dificultad.

La decisión metodológica derivada de esta sección es conservar `M0_historical_profile` como modelo principal y cartográfico conservador para la dificultad SER, porque mantiene mejor el comportamiento en valores altos del proxy y evita una recalibración que suaviza episodios relevantes. `M1a_ridge_profile_only` se conserva como modelo secundario de contraste, ya que demuestra que una recalibración lineal del perfil histórico mejora el error medio global, aunque no el objetivo más crítico de presión alta.

A partir de estos resultados, no se amplía la comparación a estimadores adicionales como ElasticNet. Las variantes Ridge con más variables no muestran una mejora robusta frente al baseline, y el principal problema detectado no es la selección de variables, sino la tensión entre mejorar el error medio global y preservar los valores altos del target. Una ampliación de modelos tendría sentido como línea futura, por ejemplo mediante pérdidas ponderadas o enfoques específicos para presión alta, pero no como parte del core de este notebook.


## 8. Controles de leakage y validación técnica

Esta sección consolida los controles técnicos generados a lo largo del notebook. Su objetivo no es volver a entrenar modelos, sino comprobar de forma agregada que el dataset, los splits, las features, las predicciones y las métricas respetan el contrato metodológico definido al inicio.

Los controles se organizan en varios bloques:

* validaciones del panel base: existencia del target, granularidad de 30 minutos, conversión temporal y ausencia de duplicados por `barrio_key × intervalo_inicio`;
* validaciones de splits: tramos de entrenamiento y evaluación no vacíos, ausencia de solapamiento y orden temporal correcto;
* validaciones de features: exclusión del target y de métricas SER contemporáneas del intervalo objetivo;
* validaciones del baseline: ajuste exclusivo con train y predicciones completas sobre evaluación;
* validaciones del modelo mínimo: construcción de la feature histórica sin predicción in-sample, exclusión de filas sin histórico interno en train y predicciones completas sobre evaluación;
* validaciones de evaluación: predicciones finitas, presencia del baseline y de las variantes del modelo mínimo en cada split, ausencia de duplicados y cálculo del umbral de target alto únicamente con train.

Los checks con estado `FAIL` bloquean la continuación del notebook. Los checks con estado `WARNING` no bloquean la ejecución, pero deben revisarse antes de interpretar resultados completos o activar escritura de salidas. En modo `smoke`, estos controles solo verifican la arquitectura del pipeline; la interpretación final queda condicionada a la ejecución completa. La tabla completa de checks queda disponible en memoria como `technical_validation_checks`; por defecto solo se muestran el estado agregado, el resumen por grupo y los checks con `WARNING` o `FAIL` para evitar saturar el notebook.

In [19]:
def add_technical_check(records: list[dict], check_group: str, check: str, status: str, detail: str) -> None:
    allowed_statuses = {"OK", "WARNING", "FAIL"}
    if status not in allowed_statuses:
        raise ValueError(f"Estado de check no válido: {status!r}")
    records.append({
        "check_group": check_group,
        "check": check,
        "status": status,
        "detail": detail,
    })


technical_check_records = []
expected_minimum_model_ids = [
    "M1a_ridge_profile_only",
    "M1b_ridge_profile_season_structural",
    "M1c_ridge_full_exante",
]

# Panel checks
target_in_model_base = "model_base" in globals() and TARGET_COLUMN in model_base.columns
add_technical_check(technical_check_records, "panel", "target_column_present", "OK" if target_in_model_base else "FAIL", f"target={TARGET_COLUMN}")
key_columns_missing = [col for col in KEY_COLUMNS if "model_base" not in globals() or col not in model_base.columns]
add_technical_check(technical_check_records, "panel", "key_columns_present", "OK" if not key_columns_missing else "FAIL", f"missing={key_columns_missing}")
if target_in_model_base:
    model_base_target = pd.to_numeric(model_base[TARGET_COLUMN], errors="coerce")
    target_missing = model_base_target.isna().sum()
    target_non_finite = (~np.isfinite(model_base_target.to_numpy(dtype=float))).sum()
    target_negative = model_base_target.lt(0).sum()
else:
    target_missing = target_non_finite = target_negative = pd.NA
add_technical_check(technical_check_records, "panel", "target_not_null", "OK" if target_in_model_base and target_missing == 0 else "FAIL", f"missing={target_missing}")
add_technical_check(technical_check_records, "panel", "target_finite", "OK" if target_in_model_base and target_non_finite == 0 else "FAIL", f"non_finite={target_non_finite}")
add_technical_check(technical_check_records, "panel", "target_non_negative", "OK" if target_in_model_base and target_negative == 0 else "FAIL", f"negative={target_negative}")
duplicate_key_rows = model_base.duplicated(KEY_COLUMNS).sum() if "model_base" in globals() and not key_columns_missing else pd.NA
add_technical_check(technical_check_records, "panel", "unique_barrio_interval", "OK" if duplicate_key_rows == 0 else "FAIL", f"duplicate_rows={duplicate_key_rows}")
panel_fail_count = panel_column_checks["status"].eq("FAIL").sum() if "panel_column_checks" in globals() else pd.NA
add_technical_check(technical_check_records, "panel", "panel_column_checks_no_fail", "OK" if panel_fail_count == 0 else "FAIL", f"fail_count={panel_fail_count}")

# Split checks
split_fail_count = split_checks["status"].eq("FAIL").sum() if "split_checks" in globals() else pd.NA
temporal_split_fail_count = temporal_leakage_split_checks["status"].eq("FAIL").sum() if "temporal_leakage_split_checks" in globals() else pd.NA
split_indices_available = "split_indices" in globals() and bool(split_indices)
add_technical_check(technical_check_records, "splits", "split_checks_no_fail", "OK" if split_fail_count == 0 else "FAIL", f"fail_count={split_fail_count}")
add_technical_check(technical_check_records, "splits", "temporal_leakage_split_checks_no_fail", "OK" if temporal_split_fail_count == 0 else "FAIL", f"fail_count={temporal_split_fail_count}")
add_technical_check(technical_check_records, "splits", "split_indices_available", "OK" if split_indices_available else "FAIL", f"available={split_indices_available}")

# Feature checks
feature_sets_available = "minimum_model_feature_sets" in globals() and bool(minimum_model_feature_sets) and all(bool(features) for features in minimum_model_feature_sets.values())
feature_set_sizes = {model_id: len(features) for model_id, features in minimum_model_feature_sets.items()} if "minimum_model_feature_sets" in globals() else {}
add_technical_check(technical_check_records, "features", "minimum_model_feature_columns_available", "OK" if feature_sets_available else "FAIL", f"feature_set_sizes={feature_set_sizes}")

target_feature_overlap = {
    model_id: [feature for feature in features if feature == TARGET_COLUMN]
    for model_id, features in minimum_model_feature_sets.items()
} if "minimum_model_feature_sets" in globals() else {"unavailable": [TARGET_COLUMN]}
target_feature_overlap = {model_id: overlap for model_id, overlap in target_feature_overlap.items() if overlap}
add_technical_check(technical_check_records, "features", "target_not_in_minimum_model_features", "OK" if not target_feature_overlap else "FAIL", f"overlap={target_feature_overlap}")

forbidden_feature_overlap = {
    model_id: sorted(set(features) & set(available_forbidden_feature_columns))
    for model_id, features in minimum_model_feature_sets.items()
} if "minimum_model_feature_sets" in globals() and "available_forbidden_feature_columns" in globals() else {"unavailable": ["unavailable"]}
forbidden_feature_overlap = {model_id: overlap for model_id, overlap in forbidden_feature_overlap.items() if overlap}
add_technical_check(technical_check_records, "features", "forbidden_columns_not_in_minimum_model_features", "OK" if not forbidden_feature_overlap else "FAIL", f"overlap={forbidden_feature_overlap}")

leakage_feature_overlap = {
    model_id: sorted(set(features) & set(LEAKAGE_COLUMNS))
    for model_id, features in minimum_model_feature_sets.items()
} if "minimum_model_feature_sets" in globals() else {"unavailable": ["unavailable"]}
leakage_feature_overlap = {model_id: overlap for model_id, overlap in leakage_feature_overlap.items() if overlap}
add_technical_check(technical_check_records, "features", "leakage_columns_not_in_minimum_model_features", "OK" if not leakage_feature_overlap else "FAIL", f"overlap={leakage_feature_overlap}")

feature_summary_exists = "minimum_model_feature_summary" in globals() and {"model_id", "feature"}.issubset(minimum_model_feature_summary.columns)
expected_feature_pairs = {
    (model_id, feature)
    for model_id, features in minimum_model_feature_sets.items()
    for feature in features
} if "minimum_model_feature_sets" in globals() else set()
actual_feature_pairs = set(zip(minimum_model_feature_summary["model_id"], minimum_model_feature_summary["feature"])) if feature_summary_exists else set()
feature_summary_complete = feature_summary_exists and actual_feature_pairs == expected_feature_pairs and len(minimum_model_feature_summary) == len(expected_feature_pairs)
add_technical_check(technical_check_records, "features", "feature_summary_complete", "OK" if feature_summary_complete else "FAIL", f"summary_rows={len(minimum_model_feature_summary) if 'minimum_model_feature_summary' in globals() else pd.NA}; expected_rows={len(expected_feature_pairs)}")
feature_summary_has_truncation = any("..." in str(value) for value in minimum_model_feature_summary["feature"]) if feature_summary_exists else True
add_technical_check(technical_check_records, "features", "feature_summary_not_truncated", "OK" if feature_summary_exists and not feature_summary_has_truncation else "FAIL", f"has_truncation={feature_summary_has_truncation}")

# Baseline checks
baseline_fail_count = baseline_checks["status"].eq("FAIL").sum() if "baseline_checks" in globals() else pd.NA
baseline_predictions_available = "baseline_predictions" in globals() and not baseline_predictions.empty
baseline_model_present = baseline_predictions_available and "M0_historical_profile" in set(baseline_predictions["model_id"])
add_technical_check(technical_check_records, "baseline", "baseline_checks_no_fail", "OK" if baseline_fail_count == 0 else "FAIL", f"fail_count={baseline_fail_count}")
add_technical_check(technical_check_records, "baseline", "baseline_predictions_available", "OK" if baseline_predictions_available else "FAIL", f"n_rows={len(baseline_predictions) if 'baseline_predictions' in globals() else pd.NA}")
add_technical_check(technical_check_records, "baseline", "baseline_model_present", "OK" if baseline_model_present else "FAIL", f"present={baseline_model_present}")

# Minimum model checks
minimum_model_fail_count = minimum_model_checks["status"].eq("FAIL").sum() if "minimum_model_checks" in globals() else pd.NA
minimum_model_predictions_available = "minimum_model_predictions" in globals() and not minimum_model_predictions.empty
minimum_model_ids_present = sorted(set(minimum_model_predictions["model_id"])) if minimum_model_predictions_available else []
missing_minimum_model_ids = [model_id for model_id in expected_minimum_model_ids if model_id not in minimum_model_ids_present]
minimum_model_history_feature_used = "minimum_model_feature_sets" in globals() and all("historical_profile_pred" in features for features in minimum_model_feature_sets.values())
add_technical_check(technical_check_records, "minimum_model", "minimum_model_checks_no_fail", "OK" if minimum_model_fail_count == 0 else "FAIL", f"fail_count={minimum_model_fail_count}")
add_technical_check(technical_check_records, "minimum_model", "minimum_model_predictions_available", "OK" if minimum_model_predictions_available else "FAIL", f"n_rows={len(minimum_model_predictions) if 'minimum_model_predictions' in globals() else pd.NA}")
add_technical_check(technical_check_records, "minimum_model", "minimum_model_present", "OK" if not missing_minimum_model_ids else "FAIL", f"missing={missing_minimum_model_ids}; present={minimum_model_ids_present}")
add_technical_check(technical_check_records, "minimum_model", "minimum_model_history_feature_used", "OK" if minimum_model_history_feature_used else "FAIL", f"used_all={minimum_model_history_feature_used}")

# Evaluation checks
evaluation_fail_count = evaluation_checks["status"].eq("FAIL").sum() if "evaluation_checks" in globals() else pd.NA
evaluation_predictions_available = "evaluation_predictions" in globals() and not evaluation_predictions.empty
metrics_comparison_available = "metrics_comparison" in globals() and not metrics_comparison.empty
model_comparison_delta_available = "model_comparison_delta" in globals() and not model_comparison_delta.empty
evaluation_duplicate_count = evaluation_predictions.duplicated(["model_id", "split_id", "barrio_key", "intervalo_inicio"]).sum() if evaluation_predictions_available else pd.NA
high_target_thresholds_available = "high_target_thresholds" in globals() and not high_target_thresholds.empty and np.isfinite(high_target_thresholds["target_high_threshold"].to_numpy(dtype=float)).all()
skill_columns = {
    "mae_skill_vs_baseline",
    "rmse_skill_vs_baseline",
    "mae_high_target_skill_vs_baseline",
    "rmse_high_target_skill_vs_baseline",
}
skill_columns_available = metrics_comparison_available and skill_columns.issubset(metrics_comparison.columns)
delta_columns = {
    "delta_mae_high_target",
    "delta_rmse_high_target",
    "delta_abs_bias_high_target",
    "mae_high_target_skill_vs_baseline",
    "rmse_high_target_skill_vs_baseline",
    "preferred_model_by_high_target_mae",
}
delta_columns_available = model_comparison_delta_available and delta_columns.issubset(model_comparison_delta.columns)
add_technical_check(technical_check_records, "evaluation", "evaluation_checks_no_fail", "OK" if evaluation_fail_count == 0 else "FAIL", f"fail_count={evaluation_fail_count}")
add_technical_check(technical_check_records, "evaluation", "evaluation_predictions_available", "OK" if evaluation_predictions_available else "FAIL", f"n_rows={len(evaluation_predictions) if 'evaluation_predictions' in globals() else pd.NA}")
add_technical_check(technical_check_records, "evaluation", "metrics_comparison_available", "OK" if metrics_comparison_available else "FAIL", f"n_rows={len(metrics_comparison) if 'metrics_comparison' in globals() else pd.NA}")
add_technical_check(technical_check_records, "evaluation", "model_comparison_delta_available", "OK" if model_comparison_delta_available else "FAIL", f"n_rows={len(model_comparison_delta) if 'model_comparison_delta' in globals() else pd.NA}")
add_technical_check(technical_check_records, "evaluation", "no_duplicate_evaluation_predictions", "OK" if evaluation_duplicate_count == 0 else "FAIL", f"duplicate_rows={evaluation_duplicate_count}")
add_technical_check(technical_check_records, "evaluation", "high_target_thresholds_available", "OK" if high_target_thresholds_available else "FAIL", f"available={high_target_thresholds_available}")
add_technical_check(technical_check_records, "evaluation", "skill_columns_available", "OK" if skill_columns_available else "FAIL", f"required={sorted(skill_columns)}")
add_technical_check(technical_check_records, "evaluation", "high_target_delta_columns_available", "OK" if delta_columns_available else "FAIL", f"required={sorted(delta_columns)}")

# Execution mode checks
run_mode_valid = RUN_MODE in {"smoke", "full"}
smoke_without_writes = RUN_MODE != "smoke" or WRITE_OUTPUTS is False
full_write_detail = "not full mode"
if RUN_MODE == "full" and WRITE_OUTPUTS is False:
    full_write_detail = "no se escribirán outputs"
elif RUN_MODE == "full" and WRITE_OUTPUTS is True:
    full_write_detail = "la escritura se controlará en Sección 10"
add_technical_check(technical_check_records, "execution_mode", "run_mode_valid", "OK" if run_mode_valid else "FAIL", f"RUN_MODE={RUN_MODE}")
add_technical_check(technical_check_records, "execution_mode", "smoke_without_writes", "OK" if smoke_without_writes else "FAIL", f"RUN_MODE={RUN_MODE}; WRITE_OUTPUTS={WRITE_OUTPUTS}")
add_technical_check(technical_check_records, "execution_mode", "full_write_guard", "OK", full_write_detail)

warning_sources = [
    ("panel", "panel_column_checks_warning_count", panel_column_checks if "panel_column_checks" in globals() else None),
    ("splits", "split_checks_warning_count", split_checks if "split_checks" in globals() else None),
    ("baseline", "baseline_checks_warning_count", baseline_checks if "baseline_checks" in globals() else None),
    ("minimum_model", "minimum_model_checks_warning_count", minimum_model_checks if "minimum_model_checks" in globals() else None),
    ("evaluation", "evaluation_checks_warning_count", evaluation_checks if "evaluation_checks" in globals() else None),
]
for check_group, check_name, checks_df in warning_sources:
    warning_count = checks_df["status"].eq("WARNING").sum() if checks_df is not None else 0
    add_technical_check(technical_check_records, check_group, check_name, "OK" if warning_count == 0 else "WARNING", f"warning_count={warning_count}")

technical_validation_checks = pd.DataFrame(technical_check_records)
technical_validation_summary = (
    technical_validation_checks.assign(
        is_ok=lambda df: df["status"].eq("OK"),
        is_warning=lambda df: df["status"].eq("WARNING"),
        is_fail=lambda df: df["status"].eq("FAIL"),
    )
    .groupby("check_group", as_index=False)
    .agg(
        n_checks=("check", "count"),
        n_ok=("is_ok", "sum"),
        n_warning=("is_warning", "sum"),
        n_fail=("is_fail", "sum"),
    )
)
technical_validation_summary["group_status"] = np.select(
    [technical_validation_summary["n_fail"].gt(0), technical_validation_summary["n_warning"].gt(0)],
    ["FAIL", "WARNING"],
    default="OK",
)

total_checks = len(technical_validation_checks)
total_ok = technical_validation_checks["status"].eq("OK").sum()
total_warning = technical_validation_checks["status"].eq("WARNING").sum()
total_fail = technical_validation_checks["status"].eq("FAIL").sum()
overall_status = "FAIL" if total_fail > 0 else "WARNING" if total_warning > 0 else "OK"
technical_validation_status = pd.DataFrame([
    {
        "run_mode": RUN_MODE,
        "n_checks": total_checks,
        "n_ok": total_ok,
        "n_warning": total_warning,
        "n_fail": total_fail,
        "overall_status": overall_status,
        "can_continue": total_fail == 0,
    }
])

display(technical_validation_status)
display(technical_validation_summary)

technical_validation_checks_with_issues = technical_validation_checks.loc[technical_validation_checks["status"].isin(["WARNING", "FAIL"])]
if SHOW_DEBUG_TABLES:
    display(technical_validation_checks)
elif not technical_validation_checks_with_issues.empty:
    display(technical_validation_checks_with_issues)

failed_technical_validation_checks = technical_validation_checks.loc[technical_validation_checks["status"].eq("FAIL")]
if not failed_technical_validation_checks.empty:
    if not SHOW_DEBUG_TABLES:
        display(technical_validation_checks_with_issues)
    raise ValueError(f"Checks técnicos fallidos: {failed_technical_validation_checks[['check_group', 'check']].to_dict('records')}")


,run_mode,n_checks,n_ok,n_warning,n_fail,overall_status,can_continue
0,full,39,38,1,0,WARNING,True


,check_group,n_checks,n_ok,n_warning,n_fail,group_status
0,baseline,4,4,0,0,OK
1,evaluation,9,9,0,0,OK
2,execution_mode,3,3,0,0,OK
3,features,6,6,0,0,OK
4,minimum_model,5,5,0,0,OK
5,panel,8,8,0,0,OK
6,splits,4,3,1,0,WARNING


,check_group,check,status,detail
35,splits,split_checks_warning_count,WARNING,warning_count=1


La validación técnica consolida 39 comprobaciones del pipeline, de las cuales 38 finalizan en estado `OK` y una en estado `WARNING`. No aparece ningún `FAIL`, por lo que el notebook puede continuar y los resultados son técnicamente interpretables.

El único warning procede del bloque de splits y corresponde a la advertencia ya identificada en la Sección 3: dos barrios aparecen en la evaluación de 2025 sin estar presentes en el entrenamiento 2023-2024. Esta incidencia no invalida la ejecución, porque el baseline dispone de niveles de fallback y porque el split final de test no presenta barrios nuevos. No obstante, debe mantenerse como limitación metodológica de la partición de validación.

El resto de bloques queda validado: el panel no presenta problemas críticos, las features excluyen el target y las métricas contemporáneas del intervalo, las predicciones de M0 y M1 son completas y finitas, las métricas comparativas se generan correctamente y el modo de ejecución respeta la política de escritura protegida. En conjunto, la sección permite cerrar el notebook desde el punto de vista técnico antes de interpretar las salidas y decidir qué artefactos se guardan.


## 9. Selección y ajuste final del modelo M0

La Sección 7 permite seleccionar `M0_historical_profile` como modelo principal conservador para el bloque SER. 

Una vez seleccionada la especificación, el modelo final se ajusta con todo el histórico observado disponible en `model_base`, desde `2023-01-02 09:00:00` hasta `2026-03-31 20:30:00`. Este ajuste final no forma parte de la evaluación temporal, sino de la generación del artefacto reutilizable para escenarios posteriores. Por ese motivo, no se calculan métricas nuevas contra el propio histórico utilizado para ajustar el modelo, ya que serían métricas in-sample y podrían inducir una lectura artificialmente optimista.

En esta formulación, el modelo M0 no se guarda como un objeto Python serializado, sino como una tabla de perfiles históricos en formato largo. Esta decisión es más trazable y menos dependiente del entorno de ejecución: el modelo queda definido por medias históricas del target y una jerarquía explícita de fallback.

La jerarquía del modelo seleccionado es:

1. `barrio_key × dia_semana_num × intervalo_30min_id`;
2. `barrio_key × intervalo_30min_id`;
3. `barrio_key`;
4. `dia_semana_num × intervalo_30min_id`;
5. media global.

Cada fila de `ser_m0_selected_profiles` contiene el nivel de perfil, el orden de fallback, la predicción media histórica, el número de observaciones usadas y el periodo de entrenamiento. El fichero `ser_m0_selected_model_metadata.json` conserva la trazabilidad del artefacto, el target, la unidad analítica, el periodo de entrenamiento, la jerarquía de fallback y las limitaciones de uso.

In [20]:
selected_model_id = "M0_historical_profile"
selected_model_role = "modelo_principal_conservador_ser"
selected_training_df = model_base.copy()
selected_training_df["intervalo_inicio"] = pd.to_datetime(selected_training_df["intervalo_inicio"], errors="coerce")

selected_required_columns = [
    "barrio_key",
    "dia_semana_num",
    "intervalo_30min_id",
    "intervalo_inicio",
    TARGET_COLUMN,
]
missing_selected_columns = [col for col in selected_required_columns if col not in selected_training_df.columns]
if missing_selected_columns:
    raise ValueError(f"Faltan columnas para construir el M0 seleccionado: {missing_selected_columns}")

selected_training_period_start = selected_training_df["intervalo_inicio"].min()
selected_training_period_end = selected_training_df["intervalo_inicio"].max()
expected_selected_training_period_end = pd.Timestamp("2026-03-31 20:30:00")
if selected_training_period_end != expected_selected_training_period_end:
    raise ValueError(
        "El periodo final de entrenamiento del M0 seleccionado no coincide con el esperado: "
        f"observado={selected_training_period_end}; esperado={expected_selected_training_period_end}"
    )

selected_training_target = pd.to_numeric(selected_training_df[TARGET_COLUMN], errors="coerce")
selected_training_finite_mask = selected_training_target.notna() & np.isfinite(selected_training_target.to_numpy(dtype=float))
selected_training_finite = selected_training_df.loc[selected_training_finite_mask].copy()
selected_training_finite[TARGET_COLUMN] = selected_training_target.loc[selected_training_finite_mask].to_numpy(dtype=float)

training_period_start_str = str(selected_training_period_start)
training_period_end_str = str(selected_training_period_end)
selected_profile_specs = [
    {"profile_level": "barrio_dow_interval", "fallback_order": 1, "keys": ["barrio_key", "dia_semana_num", "intervalo_30min_id"]},
    {"profile_level": "barrio_interval", "fallback_order": 2, "keys": ["barrio_key", "intervalo_30min_id"]},
    {"profile_level": "barrio", "fallback_order": 3, "keys": ["barrio_key"]},
    {"profile_level": "dow_interval", "fallback_order": 4, "keys": ["dia_semana_num", "intervalo_30min_id"]},
]


def build_selected_profile_frame(spec: dict) -> pd.DataFrame:
    profile = (
        selected_training_finite
        .groupby(spec["keys"], dropna=False)[TARGET_COLUMN]
        .agg(profile_prediction="mean", n_train_obs="count")
        .reset_index()
    )
    profile["model_id"] = selected_model_id
    profile["model_role"] = selected_model_role
    profile["profile_level"] = spec["profile_level"]
    profile["fallback_order"] = spec["fallback_order"]
    return profile


selected_profile_frames = [build_selected_profile_frame(spec) for spec in selected_profile_specs]
global_profile = pd.DataFrame([
    {
        "model_id": selected_model_id,
        "model_role": selected_model_role,
        "profile_level": "global_mean",
        "fallback_order": 5,
        "profile_prediction": selected_training_finite[TARGET_COLUMN].mean(),
        "n_train_obs": int(selected_training_finite[TARGET_COLUMN].count()),
    }
])

ser_m0_selected_profiles = pd.concat([*selected_profile_frames, global_profile], ignore_index=True, sort=False)
for nullable_key in ["barrio_key", "dia_semana_num", "intervalo_30min_id"]:
    if nullable_key not in ser_m0_selected_profiles.columns:
        ser_m0_selected_profiles[nullable_key] = pd.NA

ser_m0_selected_profiles["target_column"] = TARGET_COLUMN
ser_m0_selected_profiles["granularidad_min"] = 30
ser_m0_selected_profiles["training_period_start"] = training_period_start_str
ser_m0_selected_profiles["training_period_end"] = training_period_end_str
ser_m0_selected_profiles = ser_m0_selected_profiles[[
    "model_id",
    "model_role",
    "profile_level",
    "fallback_order",
    "barrio_key",
    "dia_semana_num",
    "intervalo_30min_id",
    "profile_prediction",
    "n_train_obs",
    "target_column",
    "granularidad_min",
    "training_period_start",
    "training_period_end",
]].sort_values(["fallback_order", "barrio_key", "dia_semana_num", "intervalo_30min_id"], na_position="last").reset_index(drop=True)
ser_m0_selected_profiles["n_train_obs"] = ser_m0_selected_profiles["n_train_obs"].astype("int64")

fallback_hierarchy = [
    {"fallback_order": spec["fallback_order"], "profile_level": spec["profile_level"], "keys": spec["keys"]}
    for spec in selected_profile_specs
] + [{"fallback_order": 5, "profile_level": "global_mean", "keys": []}]

ser_m0_selected_model_metadata = {
    "model_id": selected_model_id,
    "model_role": selected_model_role,
    "target_column": TARGET_COLUMN,
    "unit": "ratio de ocupacion pagada proxy sobre capacidad-tiempo SER observable",
    "granularity_min": "30",
    "training_period_start": training_period_start_str,
    "training_period_end": training_period_end_str,
    "n_training_rows": str(len(selected_training_df)),
    "n_training_barrios": str(selected_training_df["barrio_key"].nunique(dropna=True)),
    "source_panel_path": relpath(PANEL_FINAL_PATH),
    "created_by_notebook": "notebooks/04_03_ser_model_dataset_baseline.ipynb",
    "python_version": sys.version.split()[0],
    "pandas_version": pd.__version__,
    "pyarrow_version": pa.__version__,
    "recommended_environment": "tfm-parking",
    "parquet_write_version": "1.0",
    "parquet_data_page_version": "1.0",
    "profile_levels": [item["profile_level"] for item in fallback_hierarchy],
    "fallback_hierarchy": fallback_hierarchy,
    "selected_after_validation": "Seccion 7; seleccion basada en splits temporales sin recalcular metricas in-sample",
    "selection_rationale": "M0 se conserva como modelo principal conservador por su comportamiento en valores altos del target y por evitar recalibraciones lineales no robustas.",
    "future_use": "Artefacto reutilizable para 04_04_ser_prediccion_escenarios.ipynb, mapas posteriores y posibles funciones en src.",
    "limitations": "Proxy de presion pagada SER; no mide ocupacion real total ni disponibilidad plaza a plaza; perfiles historicos sin variables contemporaneas futuras.",
}

ser_m0_selected_profiles_summary = (
    ser_m0_selected_profiles
    .groupby(["profile_level", "fallback_order"], as_index=False)
    .agg(
        n_profiles=("profile_prediction", "size"),
        total_train_obs=("n_train_obs", "sum"),
        profile_prediction_min=("profile_prediction", "min"),
        profile_prediction_mean=("profile_prediction", "mean"),
        profile_prediction_max=("profile_prediction", "max"),
    )
    .sort_values("fallback_order")
)

selected_expected_levels = ["barrio_dow_interval", "barrio_interval", "barrio", "dow_interval", "global_mean"]
selected_profile_prediction = pd.to_numeric(ser_m0_selected_profiles["profile_prediction"], errors="coerce")
selected_profile_prediction_finite = np.isfinite(selected_profile_prediction.to_numpy(dtype=float))
ser_m0_selected_model_check_records = [
    {"check": "ser_m0_selected_profiles_exists_not_empty", "status": "OK" if "ser_m0_selected_profiles" in globals() and not ser_m0_selected_profiles.empty else "FAIL", "detail": f"n_rows={len(ser_m0_selected_profiles) if 'ser_m0_selected_profiles' in globals() else 0}"},
    {"check": "expected_profile_levels_present", "status": "OK" if set(selected_expected_levels).issubset(set(ser_m0_selected_profiles["profile_level"])) else "FAIL", "detail": f"levels={sorted(ser_m0_selected_profiles['profile_level'].dropna().unique().tolist())}"},
    {"check": "global_mean_single_row", "status": "OK" if ser_m0_selected_profiles["profile_level"].eq("global_mean").sum() == 1 else "FAIL", "detail": f"n_global_mean={int(ser_m0_selected_profiles['profile_level'].eq('global_mean').sum())}"},
    {"check": "profile_prediction_not_null", "status": "OK" if selected_profile_prediction.notna().all() else "FAIL", "detail": f"missing={int(selected_profile_prediction.isna().sum())}"},
    {"check": "profile_prediction_finite", "status": "OK" if selected_profile_prediction_finite.all() else "FAIL", "detail": f"non_finite={int((~selected_profile_prediction_finite).sum())}"},
    {"check": "n_train_obs_positive", "status": "OK" if ser_m0_selected_profiles["n_train_obs"].gt(0).all() else "FAIL", "detail": f"non_positive={int(ser_m0_selected_profiles['n_train_obs'].le(0).sum())}"},
    {"check": "training_period_start_matches_model_base", "status": "OK" if ser_m0_selected_profiles["training_period_start"].eq(str(model_base["intervalo_inicio"].min())).all() else "FAIL", "detail": f"start={training_period_start_str}; model_base_min={model_base['intervalo_inicio'].min()}"},
    {"check": "training_period_end_matches_model_base", "status": "OK" if ser_m0_selected_profiles["training_period_end"].eq(str(model_base["intervalo_inicio"].max())).all() else "FAIL", "detail": f"end={training_period_end_str}; model_base_max={model_base['intervalo_inicio'].max()}"},
    {"check": "ser_m0_selected_model_metadata_exists", "status": "OK" if "ser_m0_selected_model_metadata" in globals() and isinstance(ser_m0_selected_model_metadata, dict) else "FAIL", "detail": f"type={type(ser_m0_selected_model_metadata).__name__ if 'ser_m0_selected_model_metadata' in globals() else 'missing'}"},
    {"check": "metadata_model_id_matches", "status": "OK" if ser_m0_selected_model_metadata.get("model_id") == selected_model_id else "FAIL", "detail": f"metadata_model_id={ser_m0_selected_model_metadata.get('model_id')}"},
]
ser_m0_selected_model_checks = pd.DataFrame(ser_m0_selected_model_check_records)
failed_ser_m0_selected_model_checks = ser_m0_selected_model_checks.loc[ser_m0_selected_model_checks["status"].eq("FAIL")]
if not failed_ser_m0_selected_model_checks.empty:
    display(ser_m0_selected_model_checks)
    raise ValueError(f"Checks del modelo M0 seleccionado fallidos: {failed_ser_m0_selected_model_checks['check'].tolist()}")

display(ser_m0_selected_profiles_summary)
display(ser_m0_selected_model_checks)


,profile_level,fallback_order,n_profiles,total_train_obs,profile_prediction_min,profile_prediction_mean,profile_prediction_max
1,barrio_dow_interval,1,8580,1292148,0.004354,0.092613,0.251665
2,barrio_interval,2,1560,1292148,0.009181,0.092521,0.222541
0,barrio,3,65,1292148,0.015096,0.092595,0.181880
3,dow_interval,4,132,1292148,0.045143,0.094592,0.111556
4,global_mean,5,1,1292148,0.094575,0.094575,0.094575


,check,status,detail
0,ser_m0_selected_profiles_exists_not_empty,OK,n_rows=10338
1,expected_profile_levels_present,OK,"levels=['barrio', 'barrio_dow_interval', 'barr..."
2,global_mean_single_row,OK,n_global_mean=1
3,profile_prediction_not_null,OK,missing=0
4,profile_prediction_finite,OK,non_finite=0
5,n_train_obs_positive,OK,non_positive=0
6,training_period_start_matches_model_base,OK,start=2023-01-02 09:00:00; model_base_min=2023...
7,training_period_end_matches_model_base,OK,end=2026-03-31 20:30:00; model_base_max=2026-0...
8,ser_m0_selected_model_metadata_exists,OK,type=dict
9,metadata_model_id_matches,OK,metadata_model_id=M0_historical_profile


El ajuste final del modelo M0 se construye correctamente sobre todo el histórico disponible del panel de modelado. El periodo de entrenamiento coincide con la cobertura completa de `model_base`, desde `2023-01-02 09:00:00` hasta `2026-03-31 20:30:00`, y los checks del artefacto final concluyen en estado `OK`.

La tabla de perfiles contiene 10.338 filas. El nivel más específico, `barrio_dow_interval`, concentra 8.580 perfiles, correspondientes a combinaciones de barrio, día de semana e intervalo de 30 minutos. Este será el primer nivel consultado en predicciones futuras. Los niveles `barrio_interval`, `barrio`, `dow_interval` y `global_mean` actúan como respaldo cuando una combinación concreta no está disponible.

La media global del proxy en el artefacto final es 0,0946, coherente con la distribución general del panel. No obstante, los perfiles específicos muestran variabilidad relevante: el nivel `barrio_dow_interval` oscila aproximadamente entre 0,0044 y 0,2517. Esto confirma que el modelo final no utiliza una media global única, sino patrones históricos diferenciados por unidad espacial y franja temporal.

La ausencia de nulos, infinitos y observaciones con `n_train_obs <= 0` permite considerar el artefacto como técnicamente válido para su uso posterior. La salida no debe interpretarse como una medición de ocupación real SER, sino como el modelo histórico seleccionado para predecir el proxy de presión pagada en escenarios posteriores.

## 10. Salidas guardadas y conexión con memoria/mapa

Esta sección define la política de escritura de salidas del notebook. Guarda artefactos de evaluación y los artefactos del modelo seleccionado creados en la Sección 9, manteniendo la escritura protegida por modo de ejecución, controles técnicos y política de sobrescritura.

El notebook no guarda salidas en modo `smoke`. En modo `full`, la escritura solo se permite si `WRITE_OUTPUTS=True`, si los controles técnicos de la Sección 8 permiten continuar, si los artefactos del modelo seleccionado existen y si `OVERWRITE_OUTPUTS` no bloquea la operación.

Las salidas candidatas son: métricas comparativas, deltas entre modelos, predicciones observadas evaluadas, perfiles históricos del M0 seleccionado y metadatos JSON de trazabilidad. No se guarda `ser_model_dataset.parquet`.

Las predicciones guardables conservan como mínimo `model_id`, `split_id`, `eval_label`, `barrio_key`, `intervalo_inicio`, `y_true`, `y_pred`, `error`, `abs_error`, `fallback_level` y variables temporales básicas necesarias para agregación posterior.

In [21]:
prediction_output_columns = [
    "model_id",
    "split_id",
    "eval_label",
    "barrio_key",
    "intervalo_inicio",
    "y_true",
    "y_pred",
    "error",
    "abs_error",
    "fallback_level",
    "hora",
]

ser_model_predictions_output = evaluation_predictions.copy()
if "hora" not in ser_model_predictions_output.columns:
    ser_model_predictions_output["hora"] = pd.to_datetime(ser_model_predictions_output["intervalo_inicio"], errors="coerce").dt.hour
if "fallback_level" not in ser_model_predictions_output.columns:
    ser_model_predictions_output["fallback_level"] = pd.NA
available_prediction_output_columns = [col for col in prediction_output_columns if col in ser_model_predictions_output.columns]
ser_model_predictions_output = ser_model_predictions_output[available_prediction_output_columns].copy()

technical_can_continue = bool(technical_validation_status["can_continue"].iloc[0]) if "technical_validation_status" in globals() and not technical_validation_status.empty else False

output_artifact_specs = [
    {
        "output_id": "ser_model_metrics_comparison",
        "object_name": "metrics_comparison",
        "path_obj": REPORTS_TABLES_DIR / "ser_model_metrics_comparison.csv",
        "output_format": "csv",
        "candidate_for_write": True,
        "use_after_04_03": "memoria",
        "write_policy": "full_only_write_outputs_true_checks_ok",
    },
    {
        "output_id": "ser_model_comparison_delta",
        "object_name": "model_comparison_delta",
        "path_obj": REPORTS_TABLES_DIR / "ser_model_comparison_delta.csv",
        "output_format": "csv",
        "candidate_for_write": True,
        "use_after_04_03": "memoria y comparación metodológica",
        "write_policy": "full_only_write_outputs_true_checks_ok",
    },
    {
        "output_id": "ser_model_predictions",
        "object_name": "ser_model_predictions_output",
        "path_obj": MODELING_DIR / "ser_model_predictions.parquet",
        "output_format": "parquet",
        "candidate_for_write": True,
        "use_after_04_03": "mapas/agregaciones posteriores si procede",
        "write_policy": "full_only_write_outputs_true_checks_ok",
    },
    {
        "output_id": "ser_m0_selected_profiles",
        "object_name": "ser_m0_selected_profiles",
        "path_obj": MODELING_DIR / "ser_m0_selected_profiles.parquet",
        "output_format": "parquet",
        "candidate_for_write": True,
        "use_after_04_03": "modelo M0 seleccionado reutilizable para escenarios y mapas posteriores",
        "write_policy": "full_only_write_outputs_true_checks_ok",
    },
    {
        "output_id": "ser_m0_selected_model_metadata",
        "object_name": "ser_m0_selected_model_metadata",
        "path_obj": MODELING_DIR / "ser_m0_selected_model_metadata.json",
        "output_format": "json",
        "candidate_for_write": True,
        "use_after_04_03": "metadatos del modelo M0 seleccionado, trazabilidad y periodo de entrenamiento",
        "write_policy": "full_only_write_outputs_true_checks_ok",
    },
]

output_write_plan = pd.DataFrame([
    {
        "output_id": spec["output_id"],
        "object_name": spec["object_name"],
        "path": relpath(spec["path_obj"]),
        "output_format": spec["output_format"],
        "candidate_for_write": spec["candidate_for_write"],
        "write_in_current_run": (
            RUN_MODE == "full"
            and WRITE_OUTPUTS is True
            and spec["candidate_for_write"] is True
            and technical_can_continue is True
        ),
        "use_after_04_03": spec["use_after_04_03"],
        "write_policy": spec["write_policy"],
    }
    for spec in output_artifact_specs
])


def add_output_write_check(records: list[dict], check: str, status: str, detail: str) -> None:
    allowed_statuses = {"OK", "WARNING", "FAIL"}
    if status not in allowed_statuses:
        raise ValueError(f"Estado de check no válido: {status!r}")
    records.append({"check": check, "status": status, "detail": detail})


output_check_records = []
run_mode_valid_for_write = RUN_MODE in {"smoke", "full"}
smoke_does_not_write = RUN_MODE != "smoke" or not output_write_plan["write_in_current_run"].any()
required_output_objects = [
    "metrics_comparison",
    "model_comparison_delta",
    "ser_model_predictions_output",
    "ser_m0_selected_profiles",
    "ser_m0_selected_model_metadata",
]
missing_required_output_objects = [name for name in required_output_objects if name not in globals()]
missing_prediction_output_columns = [col for col in prediction_output_columns if col not in ser_model_predictions_output.columns]
write_outputs_guard_ok = WRITE_OUTPUTS is True or not output_write_plan["write_in_current_run"].any()
overwrite_policy_defined = isinstance(OVERWRITE_OUTPUTS, bool)
ser_m0_selected_profiles_available = "ser_m0_selected_profiles" in globals() and isinstance(ser_m0_selected_profiles, pd.DataFrame) and not ser_m0_selected_profiles.empty
ser_m0_selected_model_metadata_available = "ser_m0_selected_model_metadata" in globals() and isinstance(ser_m0_selected_model_metadata, dict) and bool(ser_m0_selected_model_metadata)

add_output_write_check(output_check_records, "run_mode_valid_for_write", "OK" if run_mode_valid_for_write else "FAIL", f"RUN_MODE={RUN_MODE}")
add_output_write_check(output_check_records, "smoke_does_not_write", "OK" if smoke_does_not_write else "FAIL", f"write_count={int(output_write_plan['write_in_current_run'].sum())}")
add_output_write_check(output_check_records, "technical_validation_can_continue", "OK" if technical_can_continue else "FAIL", f"can_continue={technical_can_continue}")
add_output_write_check(output_check_records, "required_output_objects_available", "OK" if not missing_required_output_objects else "FAIL", f"missing={missing_required_output_objects}")
add_output_write_check(output_check_records, "ser_m0_selected_profiles_available", "OK" if ser_m0_selected_profiles_available else "FAIL", f"available={ser_m0_selected_profiles_available}")
add_output_write_check(output_check_records, "ser_m0_selected_model_metadata_available", "OK" if ser_m0_selected_model_metadata_available else "FAIL", f"available={ser_m0_selected_model_metadata_available}")
add_output_write_check(output_check_records, "predictions_output_columns_present", "OK" if not missing_prediction_output_columns else "FAIL", f"missing={missing_prediction_output_columns}")
add_output_write_check(output_check_records, "write_outputs_guard", "OK" if write_outputs_guard_ok else "FAIL", f"WRITE_OUTPUTS={WRITE_OUTPUTS}; write_count={int(output_write_plan['write_in_current_run'].sum())}")
add_output_write_check(output_check_records, "overwrite_policy_defined", "OK" if overwrite_policy_defined else "FAIL", f"OVERWRITE_OUTPUTS={OVERWRITE_OUTPUTS}")
output_write_checks = pd.DataFrame(output_check_records)

failed_output_write_checks = output_write_checks.loc[output_write_checks["status"].eq("FAIL")]
if not failed_output_write_checks.empty:
    display(output_write_checks)
    raise ValueError(f"Checks de escritura fallidos: {failed_output_write_checks['check'].tolist()}")

output_objects = {
    "metrics_comparison": metrics_comparison,
    "model_comparison_delta": model_comparison_delta,
    "ser_model_predictions_output": ser_model_predictions_output,
    "ser_m0_selected_profiles": ser_m0_selected_profiles,
    "ser_m0_selected_model_metadata": ser_m0_selected_model_metadata,
}
path_by_output_id = {spec["output_id"]: spec["path_obj"] for spec in output_artifact_specs}
output_status_records = []

for row in output_write_plan.itertuples(index=False):
    path_obj = path_by_output_id[row.output_id]
    if RUN_MODE == "smoke":
        status = "SKIPPED"
        detail = "RUN_MODE=smoke; no se escriben outputs"
    elif WRITE_OUTPUTS is False:
        status = "SKIPPED"
        detail = "WRITE_OUTPUTS=False"
    elif not row.write_in_current_run:
        status = "SKIPPED"
        detail = "artefacto no candidato para escritura en esta ejecución"
    elif path_obj.exists() and OVERWRITE_OUTPUTS is False:
        status = "SKIPPED_EXISTS"
        detail = "archivo existente y OVERWRITE_OUTPUTS=False"
    else:
        status = "PENDING_WRITE"
        detail = "pendiente de bloque protegido de escritura full"
    output_status_records.append({
        "output_id": row.output_id,
        "path": row.path,
        "write_in_current_run": row.write_in_current_run,
        "written": False,
        "status": status,
        "detail": detail,
    })

if RUN_MODE == "full" and WRITE_OUTPUTS is True:
    status_by_output_id = {record["output_id"]: record for record in output_status_records}
    for row in output_write_plan.loc[output_write_plan["write_in_current_run"]].itertuples(index=False):
        status_record = status_by_output_id[row.output_id]
        if status_record["status"] != "PENDING_WRITE":
            continue
        path_obj = path_by_output_id[row.output_id]
        path_obj.parent.mkdir(parents=True, exist_ok=True)
        if path_obj.exists():
            path_obj.unlink()
        object_to_write = output_objects[row.object_name]
        if row.output_format == "csv":
            object_to_write.to_csv(path_obj, index=False)
        elif row.output_format == "parquet":
            parquet_table = pa.Table.from_pandas(object_to_write, preserve_index=False)
            pq.write_table(parquet_table, path_obj, version="1.0", data_page_version="1.0")
            pq.ParquetFile(path_obj).read_row_group(0)
        elif row.output_format == "json":
            with path_obj.open("w", encoding="utf-8") as file:
                json.dump(object_to_write, file, ensure_ascii=False, indent=2, default=str)
        else:
            raise ValueError(f"Formato de salida no soportado en esta sección: {row.output_format}")
        status_record["written"] = True
        status_record["status"] = "WRITTEN"
        status_record["detail"] = "artefacto escrito correctamente"

output_write_status = pd.DataFrame(output_status_records)[[
    "output_id",
    "path",
    "write_in_current_run",
    "written",
    "status",
    "detail",
]]

display(output_write_plan)
display(output_write_status)

output_write_checks_with_issues = output_write_checks.loc[output_write_checks["status"].isin(["WARNING", "FAIL"])]
if SHOW_DEBUG_TABLES or not output_write_checks_with_issues.empty:
    display(output_write_checks)


,output_id,object_name,path,output_format,candidate_for_write,write_in_current_run,use_after_04_03,write_policy
0,ser_model_metrics_comparison,metrics_comparison,reports/tables/ser_model_metrics_comparison.csv,csv,True,True,memoria,full_only_write_outputs_true_checks_ok
1,ser_model_comparison_delta,model_comparison_delta,reports/tables/ser_model_comparison_delta.csv,csv,True,True,memoria y comparación metodológica,full_only_write_outputs_true_checks_ok
2,ser_model_predictions,ser_model_predictions_output,data/processed/core/ser/modeling/ser_model_pre...,parquet,True,True,mapas/agregaciones posteriores si procede,full_only_write_outputs_true_checks_ok
3,ser_m0_selected_profiles,ser_m0_selected_profiles,data/processed/core/ser/modeling/ser_m0_select...,parquet,True,True,modelo M0 seleccionado reutilizable para escen...,full_only_write_outputs_true_checks_ok
4,ser_m0_selected_model_metadata,ser_m0_selected_model_metadata,data/processed/core/ser/modeling/ser_m0_select...,json,True,True,"metadatos del modelo M0 seleccionado, trazabil...",full_only_write_outputs_true_checks_ok


,output_id,path,write_in_current_run,written,status,detail
0,ser_model_metrics_comparison,reports/tables/ser_model_metrics_comparison.csv,True,True,WRITTEN,artefacto escrito correctamente
1,ser_model_comparison_delta,reports/tables/ser_model_comparison_delta.csv,True,True,WRITTEN,artefacto escrito correctamente
2,ser_model_predictions,data/processed/core/ser/modeling/ser_model_pre...,True,True,WRITTEN,artefacto escrito correctamente
3,ser_m0_selected_profiles,data/processed/core/ser/modeling/ser_m0_select...,True,True,WRITTEN,artefacto escrito correctamente
4,ser_m0_selected_model_metadata,data/processed/core/ser/modeling/ser_m0_select...,True,True,WRITTEN,artefacto escrito correctamente


La política de escritura se ejecuta correctamente en modo `full`. Los cinco artefactos definidos como candidatos aparecen con estado `WRITTEN`: `ser_model_metrics_comparison`, `ser_model_comparison_delta`, `ser_model_predictions`, `ser_m0_selected_profiles` y `ser_m0_selected_model_metadata`.

Esta salida cierra dos necesidades distintas. Por un lado, las tablas de métricas y deltas permiten documentar en la memoria la comparación entre M0 y las variantes M1. Por otro lado, `ser_model_predictions.parquet` conserva las predicciones evaluadas sobre validación y test, útiles para análisis posteriores de error o mapas retrospectivos. Finalmente, `ser_m0_selected_profiles.parquet` y `ser_m0_selected_model_metadata.json` constituyen el artefacto reutilizable del modelo seleccionado.

La decisión de no guardar `ser_model_dataset.parquet` es coherente con el diseño reproducible del notebook. El dataset de modelado puede reconstruirse desde el panel final y no constituye por sí mismo un modelo operativo. En cambio, los perfiles M0 sí son necesarios para generar predicciones futuras por barrio, día y franja horaria sin depender de volver a ejecutar el notebook completo.

## 11. Ejemplos de predicción M0 sobre test observado

Esta sección muestra cinco ejemplos concretos de predicción del modelo M0 sobre el split de test observado (`split_test_refit`). Estos ejemplos no son escenarios futuros: proceden de la evaluación temporal ya realizada, en la que el baseline se ajusta con datos de 2023-2025 y se compara contra observaciones reales de 2026 Q1.

El objetivo de la tabla no es recalcular métricas, sino hacer tangible la mecánica de predicción del modelo seleccionado. Para cada ejemplo se muestra el barrio, el intervalo temporal, el valor observado del proxy (`y_true`), la predicción del modelo (`y_pred`), el error absoluto y el nivel de fallback utilizado.

La selección de ejemplos es determinista y cubre distintos niveles de error absoluto. Por tanto, la tabla incluye tanto casos donde el patrón histórico reproduce muy bien el valor observado como casos donde el modelo infraestima episodios de presión pagada elevada. Esta lectura es útil porque evita presentar el modelo como una solución exacta: M0 captura regularidades históricas por barrio, día de semana y franja horaria, pero no observa shocks puntuales ni cambios no explicados por el perfil histórico.

El modelo final ajustado con todo el histórico hasta marzo de 2026 no se usa en esta comparación contra test observado, ya que eso introduciría leakage retrospectivo. Ese artefacto se reserva para `04_04_ser_prediccion_escenarios.ipynb`, donde se generarán escenarios posteriores a marzo de 2026 sin `y_true` disponible.

In [22]:
m0_test_predictions_source = evaluation_predictions.loc[
    evaluation_predictions["model_id"].eq("M0_historical_profile")
    & evaluation_predictions["split_id"].eq("split_test_refit")
].copy()

if m0_test_predictions_source.empty:
    raise ValueError("No hay predicciones M0 para split_test_refit en evaluation_predictions.")

m0_test_predictions_source["abs_error_numeric"] = pd.to_numeric(
    m0_test_predictions_source["abs_error"],
    errors="coerce",
)

m0_test_predictions_source = m0_test_predictions_source.sort_values(
    ["abs_error_numeric", "barrio_key", "intervalo_inicio"],
    kind="mergesort",
).reset_index(drop=True)

example_positions = np.linspace(0, len(m0_test_predictions_source) - 1, 5).round().astype(int)
m0_test_prediction_examples = (
    m0_test_predictions_source
    .iloc[example_positions]
    .copy()
    .reset_index(drop=True)
)

# Enriquecimiento legible: nombre de barrio.
# El panel final solo conserva barrio_key; el nombre se recupera de la tabla de capacidad anual.
barrio_lookup = pd.DataFrame(columns=["barrio_key", "barrio_nombre"])
barrio_name_lookup_source = "not_available"
barrio_name_lookup_note = "no se encontró una fuente válida de nombres de barrio"
barrio_name_duplicate_keys = []

if BARRIO_CAPACITY_PATH.exists():
    capacity_schema_columns = pq.ParquetFile(BARRIO_CAPACITY_PATH).schema_arrow.names

    if {"barrio_key", "barrio"}.issubset(capacity_schema_columns):
        capacity_lookup_columns = ["barrio_key", "barrio"]
        if "anio" in capacity_schema_columns:
            capacity_lookup_columns.append("anio")

        capacity_barrio_lookup_raw = pd.read_parquet(
            BARRIO_CAPACITY_PATH,
            columns=capacity_lookup_columns,
        )

        capacity_barrio_lookup_raw = capacity_barrio_lookup_raw.dropna(
            subset=["barrio_key", "barrio"]
        ).copy()

        barrio_name_variants = (
            capacity_barrio_lookup_raw
            .drop_duplicates(["barrio_key", "barrio"])
            .groupby("barrio_key", as_index=False)
            .agg(
                n_barrio_names=("barrio", "nunique"),
                barrio_names=("barrio", lambda values: sorted(set(values))),
            )
        )
        barrio_name_duplicate_keys = barrio_name_variants.loc[
            barrio_name_variants["n_barrio_names"].gt(1),
            "barrio_key",
        ].tolist()

        # Si hay varios años o variantes de nombre, se elige una fila determinista por barrio_key.
        # La ordenación por anio prioriza el registro más reciente cuando la columna existe.
        sort_columns = ["barrio_key"]
        if "anio" in capacity_barrio_lookup_raw.columns:
            sort_columns.append("anio")
        sort_columns.append("barrio")

        barrio_lookup = (
            capacity_barrio_lookup_raw
            .sort_values(sort_columns, kind="mergesort")
            .drop_duplicates("barrio_key", keep="last")
            [["barrio_key", "barrio"]]
            .rename(columns={"barrio": "barrio_nombre"})
            .reset_index(drop=True)
        )

        barrio_name_lookup_source = relpath(BARRIO_CAPACITY_PATH)
        barrio_name_lookup_note = (
            "nombre recuperado desde ser_barrio_capacidad_anio; "
            "si existen variantes para una misma barrio_key se conserva una fila determinista"
        )

if not barrio_lookup.empty:
    m0_test_prediction_examples = m0_test_prediction_examples.merge(
        barrio_lookup,
        on="barrio_key",
        how="left",
    )
else:
    m0_test_prediction_examples["barrio_nombre"] = pd.NA

barrio_name_lookup_summary = pd.DataFrame([
    {
        "source": barrio_name_lookup_source,
        "note": barrio_name_lookup_note,
        "n_lookup_rows": len(barrio_lookup),
        "n_duplicate_name_barrio_keys": len(barrio_name_duplicate_keys),
        "duplicate_name_barrio_keys_preview": barrio_name_duplicate_keys[:10],
        "n_examples": len(m0_test_prediction_examples),
        "n_examples_with_barrio_nombre": int(m0_test_prediction_examples["barrio_nombre"].notna().sum()),
    }
])

m0_test_prediction_examples_display_columns = [
    "barrio_key",
    "barrio_nombre",
    "intervalo_inicio",
    "y_true",
    "y_pred",
    "abs_error",
    "fallback_level",
]

m0_test_prediction_examples_check_records = [
    {
        "check": "five_examples",
        "status": "OK" if len(m0_test_prediction_examples) == 5 else "FAIL",
        "detail": f"n_examples={len(m0_test_prediction_examples)}",
    },
    {
        "check": "all_have_y_true",
        "status": "OK" if m0_test_prediction_examples["y_true"].notna().all() else "FAIL",
        "detail": f"missing={int(m0_test_prediction_examples['y_true'].isna().sum())}",
    },
    {
        "check": "all_have_y_pred",
        "status": "OK" if m0_test_prediction_examples["y_pred"].notna().all() else "FAIL",
        "detail": f"missing={int(m0_test_prediction_examples['y_pred'].isna().sum())}",
    },
    {
        "check": "all_have_fallback_level",
        "status": "OK" if m0_test_prediction_examples["fallback_level"].notna().all() else "FAIL",
        "detail": f"missing={int(m0_test_prediction_examples['fallback_level'].isna().sum())}",
    },
    {
        "check": "all_split_test_refit",
        "status": "OK" if m0_test_prediction_examples["split_id"].eq("split_test_refit").all() else "FAIL",
        "detail": f"splits={sorted(m0_test_prediction_examples['split_id'].dropna().unique().tolist())}",
    },
    {
        "check": "all_m0_historical_profile",
        "status": "OK" if m0_test_prediction_examples["model_id"].eq("M0_historical_profile").all() else "FAIL",
        "detail": f"models={sorted(m0_test_prediction_examples['model_id'].dropna().unique().tolist())}",
    },
]

m0_test_prediction_examples_checks = pd.DataFrame(m0_test_prediction_examples_check_records)

failed_m0_test_prediction_examples_checks = m0_test_prediction_examples_checks.loc[
    m0_test_prediction_examples_checks["status"].eq("FAIL")
]

if not failed_m0_test_prediction_examples_checks.empty:
    display(m0_test_prediction_examples_checks)
    raise ValueError(
        "Checks de ejemplos M0 en test fallidos: "
        f"{failed_m0_test_prediction_examples_checks['check'].tolist()}"
    )

display(barrio_name_lookup_summary)
display(m0_test_prediction_examples[m0_test_prediction_examples_display_columns])
display(m0_test_prediction_examples_checks)

,source,note,n_lookup_rows,n_duplicate_name_barrio_keys,duplicate_name_barrio_keys_preview,n_examples,n_examples_with_barrio_nombre
0,data/processed/core/ser/ser_barrio_capacidad_a...,nombre recuperado desde ser_barrio_capacidad_a...,65,13,"[02_06, 03_01, 03_05, 03_06, 05_03, 05_04, 07_...",5,5


,barrio_key,barrio_nombre,intervalo_inicio,y_true,y_pred,abs_error,fallback_level
0,07_01,GAZTAMBIDE,2026-03-26 16:00:00,0.163775,0.163775,1.521182e-07,barrio_dow_interval
1,04_03,FUENTE DEL BERRO,2026-03-31 13:00:00,0.094399,0.098728,4.328596e-03,barrio_dow_interval
2,01_02,EMBAJADORES,2026-03-23 15:30:00,0.075041,0.065623,9.418751e-03,barrio_dow_interval
3,03_03,ESTRELLA,2026-03-04 13:30:00,0.096416,0.078247,1.816913e-02,barrio_dow_interval
4,15_08,ATALAYA,2026-03-11 11:00:00,0.214532,0.054527,1.600052e-01,barrio_dow_interval


,check,status,detail
0,five_examples,OK,n_examples=5
1,all_have_y_true,OK,missing=0
2,all_have_y_pred,OK,missing=0
3,all_have_fallback_level,OK,missing=0
4,all_split_test_refit,OK,splits=['split_test_refit']
5,all_m0_historical_profile,OK,models=['M0_historical_profile']


Los cinco ejemplos confirman que las predicciones evaluadas de M0 se generan correctamente sobre `split_test_refit` y que todas utilizan el nivel más específico de fallback, `barrio_dow_interval`. Esto es coherente con la cobertura del split final: el entrenamiento 2023-2025 ya contiene histórico para los 65 barrios evaluados en 2026 Q1.

La tabla muestra una escala de errores heterogénea. En los primeros ejemplos, la predicción se aproxima mucho al valor observado del proxy; en los casos intermedios, el error absoluto se mantiene en torno a magnitudes moderadas; y en el último ejemplo aparece una desviación elevada, con `y_true = 0,2145` frente a `y_pred = 0,0545`. Este último caso ilustra una limitación relevante del baseline histórico: puede infraestimar episodios concretos de presión pagada alta cuando el comportamiento del intervalo observado se aparta del patrón histórico medio.

La sección refuerza la decisión metodológica adoptada. M0 es útil como modelo conservador, interpretable y reutilizable, pero no debe presentarse como una predicción exacta de cada intervalo ni como una observación de ocupación real. Su función es estimar el valor esperado del proxy histórico para una combinación de barrio, día de semana y franja horaria. Los escenarios futuros deberán incorporar esta predicción como componente de presión pagada esperada, no como medida completa de dificultad real.

## 12. Cierre interpretativo del notebook

Este notebook cierra el bloque de modelado inicial del proxy SER a escala `barrio_key × intervalo_inicio`. A partir del panel final generado en `04_02_ser_panel_barrio_intervalo.ipynb`, se construye un dataset de modelado con granularidad de 30 minutos, se separan explícitamente target, métricas diagnósticas y variables predictoras, y se evalúan modelos bajo validación temporal. El target utilizado, `ocupacion_pagada_proxy`, no representa ocupación real de plazas SER, sino presión pagada observada históricamente respecto a la capacidad-tiempo disponible en cada barrio e intervalo.

La evaluación confirma que el patrón histórico por barrio, día de semana e intervalo horario concentra una parte relevante de la señal del proxy. El baseline `M0_historical_profile` obtiene resultados sólidos en los dos splits temporales y mantiene un comportamiento especialmente competitivo en los valores altos del target. La variante `M1a_ridge_profile_only` mejora las métricas globales de error y R², pero empeora el rendimiento en los intervalos de mayor presión pagada relativa. Dado que el objetivo del TFM es modelar dificultad de aparcamiento y no solo minimizar el error medio global, se selecciona M0 como modelo principal conservador y cartográfico. M1a queda como modelo secundario de contraste, útil para mostrar que una recalibración lineal mejora el ajuste medio, pero no preserva igual de bien los episodios altos del proxy.

Las variantes Ridge con más variables ex ante no aportan una mejora robusta. `M1b_ridge_profile_season_structural` y `M1c_ridge_full_exante` muestran ventajas parciales en algunos indicadores, pero no mejoran de forma consistente al baseline entre validación y test. Esta evidencia no justifica ampliar el core del notebook a modelos adicionales ni introducir mayor complejidad en esta fase. Posibles extensiones, como modelos no lineales ligeros, pérdidas ponderadas o enfoques específicos para presión alta, quedan como líneas futuras condicionadas a una justificación empírica adicional.

La validación técnica consolida 39 checks, con 38 en estado `OK`, un único `WARNING` y ningún `FAIL`. La advertencia corresponde a dos barrios presentes en la validación de 2025 que no aparecen en el entrenamiento 2023-2024. Esta incidencia no invalida la evaluación, porque el baseline dispone de niveles de fallback y el split final de test no presenta barrios nuevos. No obstante, debe mantenerse como limitación metodológica asociada a la ampliación temporal de la cobertura SER.

Tras seleccionar M0, el modelo final se ajusta con todo el histórico observado disponible hasta `2026-03-31 20:30:00`. Este ajuste final no se evalúa contra su propio entrenamiento para evitar una lectura in-sample artificialmente optimista. Su finalidad es generar un artefacto reutilizable para escenarios posteriores. Por ello, el modelo se persiste como perfiles históricos compactos en `ser_m0_selected_profiles.parquet` y metadatos en `ser_m0_selected_model_metadata.json`, en lugar de serializar un objeto Python o guardar una copia completa del dataset de modelado.

Las salidas guardadas cumplen dos funciones. Por un lado, `ser_model_metrics_comparison.csv` y `ser_model_comparison_delta.csv` documentan la comparación metodológica entre M0 y las variantes M1. Por otro lado, `ser_model_predictions.parquet` conserva predicciones evaluadas para análisis retrospectivos, mientras que los perfiles y metadatos del M0 seleccionado permiten generar predicciones futuras por barrio, día de semana e intervalo horario sin depender de volver a ejecutar todo el notebook.

La sección de ejemplos sobre test observado muestra que M0 produce predicciones concretas y trazables para intervalos reales de 2026 Q1. Todos los ejemplos utilizan el nivel más específico de fallback, `barrio_dow_interval`, lo que confirma que el split final dispone de cobertura histórica suficiente por barrio. Al mismo tiempo, los ejemplos evidencian una limitación relevante: el baseline puede infraestimar episodios concretos de presión pagada alta cuando el intervalo observado se aparta del patrón histórico medio.

En conjunto, el notebook deja cerrado un modelo SER inicial defendible, reproducible y alineado con las restricciones del TFM. El resultado no debe interpretarse como ocupación real ni como disponibilidad plaza a plaza, sino como predicción del valor esperado del proxy histórico de presión pagada. El siguiente paso natural es utilizar el artefacto M0 persistido en `04_04_ser_prediccion_escenarios.ipynb`, donde las predicciones posteriores a marzo de 2026 deberán presentarse como escenarios no validados con `y_true`, y podrán combinarse de forma separada con presión estructural u otros componentes contextuales para construir una lectura cartográfica de dificultad.